In [ ]:
/**
 * Key-Value Store:
 * - Key-Value Store ek non-relational database hota hai jo data ko simple (key, value)
 *   pairs ke form mein store karta hai.
 * - Example:
 *    Key          -> Value
 *    user:123     -> Mohammad
 *    user:456     -> Aslam
 *
 * - Key unique hoti hai aur uska use corresponding Value ko find/fetch karne ke liye
 *   kiya jaata hai.
 *
 * - Key plain text ho sakti hai ya internally hashed form mein bhi ho sakti hai.
 * 
 * - Value ko Key-Value Store ke liye generally opaque data maana jaata hai.
 *   Value string, list, object, etc. ho sakti hai.
 *
 *   Example:
 *      user:123 -> "Mohammad"
 *      cart:123 -> [product1, product2, product3]
 *      user:123 -> {
 *                    name: "Mohammad",
 *                    age: 30
 *                  }
 *
 * - Key-Value Store ka main focus fast lookup hota hai.
 * - Application
 *       |
 *       | Key: user:123
 *       v
 *   Key-Value Store
 *       |
 *       v
 *     Value
 * - Matlab humein Key pata hai, to us Key ke through Value ko quickly retrieve kar 
 *   sakte hain.
 *
 * - Key-Value Store ke examples:
 *   -> DynamoDB
 *   -> Redis
 *   -> Memcached
*/

/**
 * Simple words mein:
 * - Key-Value Store ko ek giant dictionary ki tarah samjho:
 *    Key          -> Value
 *    -------------------------
 *    user:123     -> User Data
 *    product:101  -> Product Data
 *    cart:123     -> Cart Data
 *
 * - Yaani iska core idea hai:
 *   "Key do -> us Key ki corresponding Value quickly mil jaayegi."
 */

In [ ]:
/**
 * Design a Distributed Key-Value Store:
 * - Ab humein ek simple Key-Value Store design karna hai jo mainly do basic operations
 *   support karega.
 *
 * 1. put(key, value)
 *    - Is operation ka use kisi Key ke saath ek Value store karne ke liye kiya jaata hai.
 *    - Example:
 *      - put("user:123", "Mohammad")
 *      - Iska matlab: user:123 -> Mohammad
 *    - Agar same Key pehle se exist karti hai, to uski Value update bhi ki ja sakti hai.
 *
 * 2. get(key)
 *    - Is operation ka use kisi Key ki corresponding Value ko retrieve karne ke liye 
 *      kiya jaata hai.
 *    - Example:
 *      - get("user:123")
 *      - Result: "Mohammad"
*/

/**
 * Basic Flow:
 *
 *       Application
 *           |
 *           v
 *    +------------------+
 *    | Key-Value Store  |
 *    +--------+---------+
 *             |
 *        +----+----+
 *        |         |
 *        v         v
 *      put()     get()
 *        |         |
 *        v         v
 *      Store    Retrieve
 *       Data      Data
*/

/**
 * Example:
 * - put("user:101", "Alice")
 * - put("user:102", "Bob")
 *
 * - get("user:101")
 *        |
 *        v
 *      "Alice"
 *
 * - get("user:102")
 *        |
 *        v
 *      "Bob"
*/

/**
 * Simple words mein:
 * a. put(key, value) -> Key ke saath Value store karo.
 * b. get(key)        -> Key ke through uski Value retrieve karo.
 *
 * Core idea:
 *
 *       Key
 *        |
 *        v
 *    +-----------+
 *    | Key-Value |
 *    |   Store   |
 *    +-----------+
 *        |
 *        v
 *      Value
 *
 * Example:
 *
 *    put("name", "Mohammad")
 *    get("name") -> "Mohammad"
 */

In [ ]:
/**
 * Design Scope:
 * - Ab hum decide karte hain ki hamare Key-Value Store ko exactly kya-kya requirements
 *   support karni chahiye.
 *
 * 1. Small Key-Value Pair
 *    - Har individual key-value pair ka size small hoga: Key + Value < 10 KB
 *    - Matlab ek single key ke against bahut large data store nahi karenge.
 *
 * 2. Ability to Store Big Data
 *    - Individual key-value pair chhota hai, lekin poora system large amount of 
 *      data store karne mein capable hona chahiye.
 *
 *       Millions / Billions of Key-Value Pairs
 *                    |
 *                    v
 *              Key-Value Store
 *
 * 3. High Availability
 *    - System highly available hona chahiye.
 *    - Agar kuch servers fail bhi ho jaayein, tab bhi system ko requests ka response
 *      dena continue karna chahiye.
 *
 *       Request
 *          |
 *          v
 *      Server 1  -> Failed
 *          X
 *
 *      Server 2  -> Available
 *          |
 *          v
 *       Response
 *
 *    - Matlab single server failure ki wajah se poora system unavailable nahi hona 
 *      chahiye.
 *
 *
 * 4. High Scalability
 *    - System ko large amount of data aur increasing traffic handle karne ke liye 
 *      horizontally scale kiya ja sake.
 *
 *       Small Traffic
 *            |
 *            v
 *         Servers
 *
 *       More Traffic
 *            |
 *            v
 *       More Servers
 *
 *    - Yaani requirement badhne par hum additional servers add karke system ki 
 *      capacity increase kar saken.
 *
 * 5. Auto Scaling
 *    - Servers ko manually add ya remove karne ki zarurat nahi honi chahiye.
 *    - Traffic ke according system automatically servers add/remove kare.
 *
 *       Traffic Increase
 *             |
 *             v
 *       Add Servers
 *
 *       Traffic Decrease
 *             |
 *             v
 *       Remove Servers
 *
 *    - Isse system changing traffic ke according automatically adjust kar sakta hai.
 *
 * 6. Tunable Consistency
 *    - Different use cases ke liye consistency ka different level required ho sakta hai.
 *    - Isliye system ko consistency level ko tune/configure karne ki flexibility deni
 *      chahiye.
 *    - Example:
 *
 *       Stronger Consistency
 *              |
 *              v
 *       Latest data preferred
 *
 *       Weaker Consistency
 *              |
 *              v
 *       Better availability / performance
 *
 *    - Yaani system ko requirement ke according consistency aur performance ke beech
 *      balance choose karne ki ability honi chahiye.
 *
 *
 * 7. Low Latency
 *    - Key-Value Store ka main goal fast read/write operations hai.
 *
 *       put(key, value)
 *              |
 *              v
 *          Fast Write
 *
 *       get(key)
 *              |
 *              v
 *          Fast Read
 *
 *    - System ko requests ka response very low latency mein dena chahiye.
*/

/**
 * Simple words mein:
 * - Hamara Key-Value Store:
 *   -> Ek key-value pair ko 10 KB se chhota rakhega.
 *   -> Overall bahut large amount of data store kar sakega.
 *   -> Server failures ke baad bhi available rahega.
 *   -> Large traffic aur data ke saath scale kar sakega.
 *   -> Traffic ke according automatically servers add/remove karega.
 *   -> Different consistency requirements support karega.
 *   -> Read aur write operations ko low latency mein perform karega.
 *
 * Core Goal:
 * - High Availability
 *          +
 * - High Scalability
 *          +
 * - Auto Scaling
 *          +
 * - Tunable Consistency
 *          +
 * - Low Latency
 *          =
 * - Reliable Large-Scale Key-Value Store
 */

In [ ]:
/**
 * Single Server Key-Value Store:
 * - Sabse pehle hum Key-Value Store ka simplest version design karte hain:
 *   ek single server par Key-Value Store.
 * - Is case mein hum data ko server ki memory mein ek in-memory hash table ke form 
 *   mein store kar sakte hain.
 *
 *        Client
 *          |
 *          v
 *     +---------+
 *     | Server  |
 *     +----+----+
 *          |
 *          v
 *    +------------+
 *    | Hash Table |
 *    +------------+
 *      |        |
 *      v        v
 *    Key A    Key B
 *      |        |
 *      v        v
 *   Value A  Value B
 *
 *
 * Hash Table ka benefit:
 * -> Key ke through Value ko very quickly find kar sakte hain.
 * -> put(key, value) fast hota hai.
 * -> get(key) bhi fast hota hai.
 *
 * Example:
 * a. put("user:101", "Alice")
 * b. Hash Table:
 *    - user:101 -> Alice
 *    - get("user:101")
 *            |
 *            v
 *          Alice
 *
 * - Lekin single-server approach ki ek major limitation hai: 
 *   server ki memory limited hoti hai.
 *
 *    Server Memory
 *    +----------------------+
 *    | Key-Value Data       |
 *    | Key-Value Data       |
 *    | Key-Value Data       |
 *    | Key-Value Data       |
 *    +----------------------+
 *              |
 *              v
 *         Memory Full
 *
 * - Agar data memory capacity se zyada ho gaya, to server additional data efficiently
 *   store nahi kar paayega.
*/

/**
 * Is problem ko reduce karne ke liye kuch optimizations kar sakte hain.
 * 1. Data Compression
 *    - Data ko compress karke memory mein kam space use kar sakte hain.
 *       Original Data
 *            |
 *            v
 *         Compress
 *            |
 *            v
 *       Smaller Data
 *    - Isse same memory mein comparatively zyada data store kiya ja sakta hai.
 *
 *
 * 2. Frequently Used Data in Memory
 *    - Har data ko memory mein rakhna zaroori nahi hai.
 *    - Frequently accessed data ko fast memory mein rakh sakte hain aur less frequently
 *      used data ko disk par move/store kar sakte hain.
 *
 *       All Data
 *          |
 *          +--------> Frequently Used -> Memory
 *          |
 *          +--------> Less Used ------> Disk
 *
 *    - Isse expensive memory ka efficient use ho sakta hai.
 *
 * Lekin in optimizations ke baad bhi single server ki fundamental limitation solve nahi
 * hoti:
 *
 *    Server Capacity
 *         |
 *         v
 *    Eventually Reached
 *
 * - Jaise-jaise data aur traffic increase hota hai, single server eventually bottleneck
 *   ban jaata hai.
 *
 * Example:
 *
 *    Small Data
 *       |
 *       v
 *    Single Server
 *       |
 *       v
 *    Works Fine
 *
 *    Large Data
 *       |
 *       v
 *    Single Server
 *       |
 *       v
 *    Capacity Reached
 *       |
 *       X
 *    Cannot Scale Further Easily
 *
 * - Isliye single-server Key-Value Store large-scale systems ke liye poorly scale 
 *    karta hai.
 * 
 * Agar humein bahut large dataset aur high traffic handle karna hai, to humein multiple
 * servers use karne padenge.
 *
 *    Client
 *       |
 *       v
 *    Multiple Servers
 *       |
 *       +--> Server 1
 *       +--> Server 2
 *       +--> Server 3
 *       +--> Server 4
 *
 * - Yaani large-scale system ke liye humein Distributed Key-Value Store design karna
 *   padega.
*/

/**
 * Simple words mein:
 * - Single Server Key-Value Store mein hum in-memory hash table ka use karke very fast
 *   get/put operations achieve kar sakte hain.
 * - Lekin server ki memory limited hoti hai, isliye data ka size eventually server ki
 *   capacity se exceed ho sakta hai.
 *
 * - Data compression aur frequently used data ko memory mein rakhne jaise optimizations
 *   help karte hain, lekin ye problem ko completely solve nahi karte.
 * - Isliye large-scale data ke liye humein next step mein Distributed Key-Value Store
 *   design karna hoga.
*/

/**
 * Core Idea:
 *
 *    Single Server
 *         |
 *         v
 *    In-Memory Hash Table
 *         |
 *         +--> Very Fast
 *         |
 *         +--> Limited by Memory
 *         |
 *         v
 *    Capacity Reached
 *         |
 *         v
 *    Distributed Key-Value Store
 */

In [ ]:
/**
 * Distributed Key-Value Store:
 * - Ab single-server Key-Value Store ki limitation samajhne ke baad, ab hum Distributed
 *   Key-Value Store design karenge.
 * - Distributed Key-Value Store mein data ek single server par nahi, balki multiple 
 *   servers par distribute hota hai.
 *
 *         Client
 *            |
 *            v
 *    +----------------+
 *    | Key-Value      |
 *    | Store Cluster  |
 *    +----------------+
 *       |    |    |
 *       v    v    v
 *    Server Server Server
 *
 * - Large-scale Distributed Key-Value Store design karte waqt humein kuch important 
 *   problems solve karni hongi:
 *    1. Data Partitioning
 *    2. Data Replication
 *    3. Consistency
 *    4. Inconsistency Resolution
 *    5. Handling Failures
 *    6. System Architecture
 *    7. Write Path
 *    8. Read Path
*/

In [ ]:

/**
 * 1. Data Partitioning:
 *
 *    Sabse pehle problem hai:
 *
 *    "Agar data bahut large hai, to ise multiple servers ke beech
 *     kaise distribute karenge?"
 *
 *    Maan lo hamare paas millions/billions of key-value pairs hain.
 *    Hum saara data ek server par nahi rakh sakte.
 *
 *    Isliye large dataset ko multiple smaller partitions mein divide
 *    karte hain aur different servers par store karte hain.
 *
 *
 *    Example:
 *
 *    Total Data
 *       |
 *       v
 *    +-----------------------+
 *    | Millions of Key-Value |
 *    |        Pairs           |
 *    +-----------+-----------+
 *                |
 *        Partitioning
 *                |
 *       +--------+--------+
 *       |        |        |
 *       v        v        v
 *    Server 1 Server 2 Server 3
 *
 *
 *    Example:
 *
 *       Server 1 -> Keys A - F
 *       Server 2 -> Keys G - M
 *       Server 3 -> Keys N - Z
 *
 *    Lekin real systems mein simply A-Z divide karna ideal nahi hai,
 *    kyunki keys evenly distributed nahi bhi ho sakti hain.
 *
 *
 *    Data Partitioning ke main challenges:
 *
 *    Challenge 1: Even Data Distribution
 *
 *       Humein ensure karna hai ki data evenly servers ke beech distribute ho.
 *
 *       Bad distribution:
 *
 *          Server 1 -> 80% Data
 *          Server 2 -> 10% Data
 *          Server 3 -> 10% Data
 *
 *       Isse Server 1 overloaded ho sakta hai.
 *
 *       Better distribution:
 *
 *          Server 1 -> ~33%
 *          Server 2 -> ~33%
 *          Server 3 -> ~34%
 *
 *
 *    Challenge 2: Minimal Data Movement
 *
 *       Ab imagine karo system mein ek new server add karna hai.
 *
 *          Before:
 *
 *          Server 1
 *          Server 2
 *          Server 3
 *
 *          After:
 *
 *          Server 1
 *          Server 2
 *          Server 3
 *          Server 4
 *
 *       Agar partitioning properly design nahi hui hai,
 *       to new server add karne par bahut saara existing data
 *       ek server se doosre server par move karna pad sakta hai.
 *
 *       Ye expensive aur time-consuming ho sakta hai.
 *
 *
 *    In challenges ko solve karne ke liye hum
 *    Consistent Hashing use kar sakte hain.
 *
 *
 * 2. Consistent Hashing:
 *
 *    Consistent Hashing ek technique hai jo keys ko servers ke
 *    beech distribute karne mein help karti hai aur server add/remove
 *    hone par data movement ko minimum rakhne ki koshish karti hai.
 *
 *    Basic idea:
 *
 *       Key
 *        |
 *        v
 *    Hash Function
 *        |
 *        v
 *    Hash Value
 *        |
 *        v
 *    Consistent Hash Ring
 *        |
 *        v
 *    Responsible Server
 *
 *
 *    Example:
 *
 *                 Server 1
 *                    |
 *              +-----+-----+
 *             /             \
 *          Server 4       Server 2
 *             \             /
 *              +-----+-----+
 *                    |
 *                 Server 3
 *
 *    Keys ko hash karke ring par place kiya jaata hai.
 *    Har key ko ring par ek responsible server milta hai.
 *
 *
 * 3. Virtual Nodes:
 *
 *    Real servers ko directly ring par ek hi position dene se
 *    distribution uneven ho sakta hai.
 *
 *    Is problem ko reduce karne ke liye har physical server ko
 *    multiple virtual nodes diye ja sakte hain.
 *
 *       Server 1
 *          |
 *       +--+--+--+
 *       |  |  |  |
 *      V1 V2 V3 V4
 *
 *       Server 2
 *          |
 *       +--+--+--+
 *       |  |  |  |
 *      V1 V2 V3 V4
 *
 *    Virtual nodes ring par multiple positions occupy karte hain.
 *
 *    Isse:
 *
 *       -> Data distribution more balanced ho sakta hai.
 *       -> Server capacity differences ko better handle kar sakte hain.
 *       -> Heterogeneous servers ko support kar sakte hain.
 *
 *    Example:
 *
 *       Powerful Server -> More Virtual Nodes
 *       Small Server    -> Fewer Virtual Nodes
 *
 *    Isse powerful server comparatively zyada data handle kar sakta hai.
 *
 *
 * 4. Automatic Scaling:
 *
 *    Consistent Hashing ka ek major benefit hai ki servers ko
 *    relatively easily add/remove kiya ja sakta hai.
 *
 *    Example:
 *
 *       Before:
 *
 *          Server 1
 *          Server 2
 *          Server 3
 *
 *       New Server:
 *
 *          Server 4
 *
 *       Server 4 add hone par sirf kuch partitions/keys ko
 *       Server 4 par move karna padta hai.
 *
 *       Humein poora dataset redistribute nahi karna padta.
 *
 *
 *    Similarly agar koi server remove/fail hota hai:
 *
 *       Server 1
 *       Server 2  X
 *       Server 3
 *
 *    To affected partitions ko available servers par
 *    redistribute kiya ja sakta hai.
 *
 *
 * 5. Why Data Partitioning is Important:
 *
 *    Without partitioning:
 *
 *       Huge Dataset
 *            |
 *            v
 *       Single Server
 *            |
 *            X
 *       Capacity Limit
 *
 *    With partitioning:
 *
 *       Huge Dataset
 *            |
 *            v
 *       +----+----+----+
 *       |    |    |    |
 *       v    v    v    v
 *      S1   S2   S3   S4
 *
 *    Ab storage capacity multiple servers mein distribute ho jaati hai.
 *
 *
 * Simple words mein:
 *
 *    Distributed Key-Value Store mein large dataset ko multiple
 *    servers ke beech divide karna padta hai.
 *
 *    Is process ko Data Partitioning kehte hain.
 *
 *    Humein do main problems solve karni hoti hain:
 *
 *       -> Data evenly distribute ho.
 *       -> Server add/remove hone par minimum data move ho.
 *
 *    Consistent Hashing in dono problems ko solve karne mein help karta hai.
 *
 *    Virtual Nodes ka use karke:
 *
 *       -> Data distribution improve hota hai.
 *       -> Different capacity wale servers support ho sakte hain.
 *       -> Scaling easier ho jaati hai.
 *
 *
 * Core Idea:
 *
 *       Large Dataset
 *            |
 *            v
 *       Data Partitioning
 *            |
 *            v
 *       Consistent Hashing
 *            |
 *            v
 *       +----+----+----+
 *       |    |    |    |
 *       v    v    v    v
 *      S1   S2   S3   S4
 *
 *    Ab data multiple servers par distribute ho gaya hai.
 *
 *    Lekin ek new problem aati hai:
 *
 *    "Agar kisi server par stored data/server hi fail ho gaya,
 *     to us data ka kya hoga?"
 *
 *    Is problem ko solve karne ke liye humein
 *    Data Replication ki zarurat padegi.
 *
 *    Next:
 *
 *       Data Partitioning
 *              |
 *              v
 *       Data Replication
 */

In [ ]:
/**
 * 1. Data Partitioning:
 *    - Jab Key-Value Store ka data bahut large ho jaata hai, to hum saara data 
 *      ek single server par store nahi kar sakte.
 *    - Isliye large dataset ko chhote-chhote partitions mein divide karke multiple 
 *      servers ke across distribute karte hain.
 *    - Example:
 *
 *          Large Dataset
 *               |
 *               v
 *        +------+------+------+
 *        |      |      |      |
 *        v      v      v      v
 *       S1     S2     S3     S4
 * 
 *       Yaani:
 *          Server 1 -> Partition 1
 *          Server 2 -> Partition 2
 *          Server 3 -> Partition 3
 *          Server 4 -> Partition 4
*/

/**
 * Ab Data Partitioning mein do main challenges aate hain:
 * 
 * 1. Challenge 1: Even Data Distribution
 *    - Humein data ko servers ke beech evenly distribute karna hai.
 *    - Bad Distribution:
 *     - Server 1 -> 70% Data
 *     - Server 2 -> 10% Data
 *     - Server 3 -> 10% Data
 *     - Server 4 -> 10% Data
 *   - Is situation mein Server 1 overloaded ho sakta hai, jabki baaki servers 
 *     underutilized rahenge.
 *
 *   - Humein ideally data ko balanced way mein distribute karna hai:
 *     - Server 1 -> ~25%
 *     - Server 2 -> ~25%
 *     - Server 3 -> ~25%
 *     - Server 4 -> ~25%
 *
 * 2. Challenge 2: Minimal Data Movement
 *    - Ab imagine karo system mein ek new server add karna hai.
 *    - Before:
 *     - Server 1
 *     - Server 2
 *     - Server 3
 *    - After:
 *     - Server 1
 *     - Server 2
 *     - Server 3
 *     - Server 4
 *   - New server add hone ke baad humein data redistribute karna padega.
 *
 *   - Agar partitioning properly design nahi hui hai, to bahut large amount of data ko
 *     servers ke beech move karna padega.
 *
 *         Old Servers
 *              |
 *              v
 *         Large Data Movement
 *              |
 *              v
 *         New Server
 *
 *   - Ye network traffic, time aur system load increase kar sakta hai.
 *
 * Note: In dono challenges ko solve karne ke liye Consistent Hashing use ki ja sakti h.
*/

/**
 * Consistent Hashing:
 * - Consistent Hashing keys ko servers ke across distribute karne mein help karti hai
 *   aur server add/remove hone par data movement ko minimum rakhti hai.
 *
 *         Key
 *          |
 *          v
 *      Hash Function
 *          |
 *          v
 *      Hash Value
 *          |
 *          v
 *    Consistent Hash Ring
 *          |
 *          v
 *       Server
 *
 * - Example:
 *       +-------------------+
 *      /                     \
 *   S1                         S2
 *    |                         |
 *    |       Hash Ring         |
 *    |                         |
 *   S4                         S3
 *      \                     /
 *       +-------------------+
 *
 *  - Har key ko hash karke ring par ek position milti hai.
 *  - Us position ke basis par key ko ek server/partition assign hota hai.
 *
 *
 * - Ab agar ek new server add hota hai:
 *   - Before: S1 ---- S2 ---- S3
 *   - After : S1 ---- S2 ---- S3 ---- S4
 *
 *   - Sirf affected keys/partitions ko move karna padta hai.
 *   - Poore dataset ko redistribute karne ki zarurat nahi hoti.
*/

/**
 * Benefits of Consistent Hashing:
 * 1. Automatic Scaling
 *    - Servers easily add/remove kiye ja sakte hain.
 *
 *          More Traffic
 *               |
 *               v
 *          Add Server
 *               |
 *               v
 *          Data Automatically Redistributed
 *
 *    - Similarly, server remove hone par affected data ko remaining servers par 
 *      redistribute kiya ja sakta hai.
 *
 * 2. Heterogeneity Support via Virtual Nodes
 *    - Har physical server ko multiple virtual nodes diye ja sakte hain.
 *
 *          Physical Server 1
 *             |
 *          +--+--+--+--+
 *          |  |  |  |  |
 *         V1 V2 V3 V4 V5
 *
 *          Physical Server 2
 *             |
 *          +--+--+--+
 *          |  |  |  |
 *         V1 V2 V3 V4
 *
 *    - Powerful server ko zyada virtual nodes de sakte hain, aur less powerful server
 *      ko fewer virtual nodes.
 *    - Example:
 *      - Powerful Server -> More Virtual Nodes
 *      - Small Server    -> Fewer Virtual Nodes
 *    - Isse different capacity wale servers ko ek hi distributed Key-Value Store mein
 *      efficiently use kiya ja sakta hai.
*/

/**
 * Simple words mein:
 * - Data Partitioning ka matlab hai large dataset ko multiple servers ke beech divide
 *   karna.
 * - Humein ensure karna hai ki:
 *   -> Data evenly distribute ho.
 *   -> Server add/remove hone par minimum data move ho.
 * - Consistent Hashing in problems ko solve karne mein help karti hai.
 *
 * - Virtual Nodes ke through:
 *   -> Data distribution better hota hai.
 *   -> Different capacity wale servers support hote hain.
 *   -> Servers ko easily add/remove kiya ja sakta hai.
*/

/**
 * Core Idea:
 *
 *   Large Dataset
 *        |
 *        v
 *   Data Partitioning
 *        |
 *        v
 *   Consistent Hashing
 *        |
 *   +----+----+----+
 *   |    |    |    |
 *   v    v    v    v
 *  S1   S2   S3   S4
 *
 * - Yaani data ko ek server par rakhne ke bajay, intelligently multiple servers mein
 *   distribute karna hi Data Partitioning ka main goal hai.
 */

In [ ]:
/**
 * 2. Data Replication:
 *
 *    Data Partitioning se humne large dataset ko multiple servers
 *    ke across distribute kar diya.
 *
 *    Lekin ab ek important problem hai:
 *
 *    "Agar jis server par data stored hai, wahi server fail ho gaya
 *     to data ka kya hoga?"
 *
 *    Is problem ko solve karne ke liye hum Data Replication use karte hain.
 *
 *
 *    Data Replication ka matlab hai same data ki multiple copies
 *    different servers par rakhna.
 *
 *         Original Data
 *              |
 *              +--------> Server 1
 *              |
 *              +--------> Server 2
 *              |
 *              +--------> Server 3
 *
 *    Maan lo replication factor N = 3 hai.
 *
 *    Matlab har key ki 3 copies system mein maintain hongi.
 *
 *
 *    Replication generally asynchronously ki ja sakti hai.
 *
 *         Client
 *            |
 *            v
 *       Primary Replica
 *            |
 *       +----+----+
 *       |         |
 *       v         v
 *    Replica 2  Replica 3
 *
 *    Primary data receive karne ke baad replicas ko
 *    asynchronously update kar sakta hai.
 *
 *    Isse write request ko har replica ke response ka wait
 *    nahi karna padta, aur system low latency maintain kar sakta hai.
 *
 *
 *    Ab question hai:
 *
 *    "Kaunsi key ki replicas kaunse servers par rakhenge?"
 *
 *    Iske liye hum Consistent Hashing ka use kar sakte hain.
 *
 *
 *    Replica Selection:
 *
 *    Step 1:
 *
 *       Key ko hash karo.
 *
 *          Key
 *           |
 *           v
 *       Hash Function
 *           |
 *           v
 *       Hash Value
 *           |
 *           v
 *       Hash Ring
 *
 *
 *    Step 2:
 *
 *       Hash ring par key ki position identify karo.
 *
 *    Step 3:
 *
 *       Key ki position se clockwise direction mein move karo.
 *
 *    Step 4:
 *
 *       Pehle N unique physical servers ko replicas ke liye select karo.
 *
 *
 *    Example:
 *
 *                    S1
 *                    |
 *             +------+------+
 *            /               \
 *          S4      KEY        S2
 *            \               /
 *             +------+------+
 *                    |
 *                   S3
 *
 *    Maan lo:
 *
 *       Replication Factor N = 3
 *
 *    Key ki position se clockwise walk karte hue:
 *
 *       First Server  -> S2
 *       Second Server -> S3
 *       Third Server  -> S4
 *
 *    To:
 *
 *       Key
 *        |
 *        +--> S2
 *        +--> S3
 *        +--> S4
 *
 *    Ye key ke 3 replicas hain.
 *
 *
 *    Virtual Nodes ke case mein ek important problem hai.
 *
 *    Same physical server ke multiple virtual nodes ring par ho sakte hain.
 *
 *       S1 -> V1
 *       S1 -> V2
 *       S1 -> V3
 *
 *    Agar hum simply first 3 ring positions select karenge,
 *    to possible hai ki:
 *
 *       V1 -> S1
 *       V2 -> S1
 *       V3 -> S1
 *
 *    Yaani technically 3 virtual nodes hain,
 *    lekin actual mein data ki 3 copies ek hi physical server par hain.
 *
 *    Ye useful replication nahi hai.
 *
 *    Isliye replicas select karte waqt humein
 *    DISTINCT PHYSICAL SERVERS choose karne hain.
 *
 *
 *    Example:
 *
 *       Ring:
 *
 *       V1(S1) -> V2(S1) -> V3(S2) -> V4(S3)
 *
 *       N = 3
 *
 *       Correct replicas:
 *
 *          S1
 *          S2
 *          S3
 *
 *       Not:
 *
 *          S1
 *          S1
 *          S2
 *
 *
 *    Ab ek aur important problem hai:
 *
 *    "Agar multiple servers same location/data center mein hain
 *     aur poora data center hi fail ho gaya to?"
 *
 *    Example:
 *
 *         Data Center 1
 *         +----------------+
 *         | S1   S2   S3   |
 *         +----------------+
 *
 *    Agar poora Data Center 1 unavailable ho gaya:
 *
 *         Data Center 1
 *                X
 *
 *    To S1, S2 aur S3 teeno replicas ek saath unavailable ho sakte hain.
 *
 *    Isko correlated failure kaha ja sakta hai.
 *
 *
 *    Isliye replicas ko different Data Centers / Availability Zones
 *    mein distribute karna better hota hai.
 *
 *
 *         Data Center 1       Data Center 2       Data Center 3
 *        +-------------+     +-------------+     +-------------+
 *        |    Server   |     |    Server   |     |    Server   |
 *        |      S1     |     |      S2     |     |      S3     |
 *        +-------------+     +-------------+     +-------------+
 *                \                 |                 /
 *                 \                |                /
 *                  +---------------+---------------+
 *                                  |
 *                                Key
 *
 *    Ab agar ek complete data center fail bhi ho jaaye:
 *
 *         Data Center 1 -> FAILED
 *         Data Center 2 -> Available
 *         Data Center 3 -> Available
 *
 *    To baaki replicas ki wajah se data available reh sakta hai.
 *
 *
 *    Complete Replication Flow:
 *
 *         Key
 *          |
 *          v
 *      Hash Function
 *          |
 *          v
 *      Consistent Hash Ring
 *          |
 *          v
 *      Clockwise Walk
 *          |
 *          v
 *      Select N Unique Physical Servers
 *          |
 *          +---------> Replica 1
 *          |
 *          +---------> Replica 2
 *          |
 *          +---------> Replica 3
 *
 *    Aur ideally ye replicas different failure domains,
 *    jaise different Data Centers / Availability Zones,
 *    mein distribute kiye jaate hain.
 *
 *
 *    Benefits:
 *
 *    -> High Availability
 *       Multiple copies available hone ki wajah se server failure
 *       ke baad bhi data access kiya ja sakta hai.
 *
 *    -> Fault Tolerance
 *       Ek ya multiple servers fail hone par replicas available
 *       reh sakte hain.
 *
 *    -> Better Durability
 *       Data ki multiple copies hone se data loss ka risk reduce hota hai.
 *
 *    -> Distributed Failure Handling
 *       Different Data Centers mein replicas rakhne se
 *       correlated failures ka impact reduce hota hai.
 *
 *
 *    Simple words mein:
 *
 *    Data Replication ka matlab hai same data ki multiple copies
 *    different physical servers par rakhna.
 *
 *    Replication factor N = 3 ka matlab hai har key ki 3 copies
 *    maintain karna.
 *
 *    Consistent Hashing ke through key ko hash ring par map karte hain,
 *    phir clockwise direction mein walk karke pehle N unique
 *    physical servers select karte hain.
 *
 *    Agar Virtual Nodes use ho rahe hain, to same physical server ko
 *    baar-baar replica ke roop mein select nahi karna chahiye.
 *
 *    Aur high availability ke liye replicas ko different
 *    Data Centers / Availability Zones mein place karna chahiye,
 *    taaki ek complete location failure hone par bhi
 *    remaining replicas data serve kar saken.
 *
 *
 *    Core Idea:
 *
 *          Key
 *           |
 *           v
 *      Consistent Hashing
 *           |
 *           v
 *      Select N Unique Servers
 *           |
 *      +----+----+----+
 *      |    |    |    |
 *      v    v    v    v
 *     S1   S2   S3   ...
 *      |    |    |
 *      v    v    v
 *     DC1  DC2  DC3
 *
 *    Yaani:
 *
 *       Partitioning -> Data ko distribute karta hai.
 *       Replication  -> Data ki multiple copies maintain karta hai.
 *
 *    Dono milkar Distributed Key-Value Store ko
 *    scalable aur highly available banate hain.
 */

In [ ]:
/**
 * 3. Data Consistency:
 *
 *    Ab humne Data Replication samajh liya.
 *    Ek key ki multiple copies different servers par stored hain.
 *
 *    Lekin ab ek important problem hai:
 *
 *    "Agar ek data ki multiple replicas hain, to ensure kaise karenge
 *     ki reads aur writes replicas ke across properly coordinate hon?"
 *
 *    Isi problem ko handle karne ke liye hum Quorum-based approach
 *    use kar sakte hain.
 *
 *
 *    Example:
 *
 *         Key: user:123
 *
 *              +---------> Server 1 -> Value = A
 *              |
 *         Key -+---------> Server 2 -> Value = A
 *              |
 *              +---------> Server 3 -> Value = A
 *
 *    Initially teeno replicas ke paas same value hai.
 *
 *    Ab client value update karta hai:
 *
 *         put(user:123, B)
 *
 *
 *    Coordinator:
 *
 *    Quorum samajhne se pehle ek important component samjho:
 *    Coordinator.
 *
 *    Client generally directly saare replicas ko manage nahi karta.
 *
 *    Client ki request pehle kisi server/node ko milti hai jo
 *    us particular request ke liye Coordinator ki tarah kaam karta hai.
 *
 *         Client
 *            |
 *            | put(key, value)
 *            v
 *      +-------------+
 *      | Coordinator |
 *      +------+------+
 *             |
 *        +----+----+
 *        |    |    |
 *        v    v    v
 *       S1   S2   S3
 *
 *    Coordinator ka kaam hai:
 *
 *       -> Key ke replicas identify karna.
 *       -> Request replicas ko bhejna.
 *       -> Write acknowledgements collect karna.
 *       -> Read responses collect karna.
 *       -> Quorum condition check karna.
 *       -> Client ko final response dena.
 *
 *    Important:
 *
 *    Coordinator necessarily ek permanent special server nahi hota.
 *    Distributed systems mein request receive karne wala node
 *    us request ke liye Coordinator ka role perform kar sakta hai.
 *
 *
 *    Quorum:
 *
 *    Quorum ka simple meaning hai:
 *
 *    "Operation successful maan-ne se pehle Coordinator ko
 *     kitne replicas ka response chahiye?"
 *
 *
 *    Yahan teen important values hain:
 *
 *         N = Total number of replicas
 *         W = Write quorum
 *         R = Read quorum
 *
 *
 *    1. N - Number of Replicas
 *
 *       N batata hai ki ek key ki total kitni copies maintain
 *       ki ja rahi hain.
 *
 *       Example:
 *
 *          N = 3
 *
 *              Coordinator
 *                   |
 *              +----+----+
 *              |    |    |
 *              v    v    v
 *             S1   S2   S3
 *
 *       Matlab key ki total 3 replicas hain.
 *
 *
 *    2. Write Quorum (W)
 *
 *       W batata hai ki Coordinator ko write successful declare
 *       karne se pehle kitne replicas ke acknowledgements chahiye.
 *
 *       Example:
 *
 *          N = 3
 *          W = 2
 *
 *
 *             Client
 *                |
 *                | PUT(key, value)
 *                v
 *          +-------------+
 *          | Coordinator |
 *          +------+------+
 *                 |
 *          +------+------+ 
 *          |      |      |
 *          v      v      v
 *         S1     S2     S3
 *          |      |      |
 *         ACK    ACK    ...
 *          |      |
 *          +---+--+
 *              |
 *              v
 *          W = 2 reached
 *              |
 *              v
 *      Write Successful
 *              |
 *              v
 *           Client
 *
 *
 *       Maan lo:
 *
 *          S1 -> ACK
 *          S2 -> ACK
 *          S3 -> Slow / No Response
 *
 *       Coordinator ke paas 2 acknowledgements aa gaye.
 *
 *       Since:
 *
 *          W = 2
 *
 *       Write successful maana ja sakta hai.
 *
 *
 *    3. Read Quorum (R)
 *
 *       R batata hai ki Coordinator ko read complete karne ke liye
 *       kitne replicas ke responses chahiye.
 *
 *       Example:
 *
 *          N = 3
 *          R = 2
 *
 *
 *             Client
 *                |
 *                | GET(key)
 *                v
 *          +-------------+
 *          | Coordinator |
 *          +------+------+
 *                 |
 *          +------+------+ 
 *          |      |      |
 *          v      v      v
 *         S1     S2     S3
 *          |      |      |
 *        Value  Value   ...
 *          |      |
 *          +---+--+
 *              |
 *              v
 *          R = 2 reached
 *              |
 *              v
 *        Return Value
 *              |
 *              v
 *           Client
 *
 *
 *       Coordinator replicas se responses collect karta hai.
 *
 *       Agar required R responses mil jaate hain,
 *       to read quorum complete ho jaata hai.
 *
 *
 *    Ab important question:
 *
 *    "R aur W ko kitna rakhen?"
 *
 *    Ye consistency aur latency ke beech tradeoff hai.
 *
 *
 *    Smaller R / W:
 *
 *       -> Coordinator ko fewer replicas ka wait karna padega.
 *       -> Lower latency.
 *       -> Operations faster ho sakte hain.
 *       -> Lekin stale/inconsistent data ka risk increase ho sakta hai.
 *
 *
 *    Larger R / W:
 *
 *       -> Coordinator ko more replicas ka response wait karna padega.
 *       -> Higher latency.
 *       -> Stronger consistency achieve karna easier hota hai.
 *
 *
 *    Strong Consistency ke liye commonly quorum overlap condition:
 *
 *                         W + R > N
 *
 *    use ki jaati hai.
 *
 *    Iska basic idea ye hai ki write quorum aur read quorum
 *    ke sets overlap karenge.
 *
 *
 *    Example:
 *
 *         N = 3
 *         W = 2
 *         R = 2
 *
 *         W + R = 2 + 2
 *               = 4
 *
 *         4 > 3
 *
 *
 *    Write:
 *
 *         Coordinator
 *              |
 *          +---+---+
 *          |       |
 *          v       v
 *         S1      S2
 *         New     New
 *
 *         S3
 *         Old
 *
 *
 *    Read:
 *
 *         Coordinator
 *              |
 *          +---+---+
 *          |       |
 *          v       v
 *         S2      S3
 *         New     Old
 *
 *    Yahan Write Quorum aur Read Quorum ka overlap S2 par hua.
 *
 *    Coordinator versions/timestamps jaise metadata ke basis par
 *    latest value determine kar sakta hai.
 *
 *
 *    Common Configuration:
 *
 *         N = 3
 *         W = 2
 *         R = 2
 *
 *    Ye consistency, availability aur latency ke beech
 *    practical balance provide kar sakta hai.
 *
 *
 *    Optimization 1: Fast Reads
 *
 *       Agar reads ko fast banana hai:
 *
 *          R = 1
 *          W = N
 *
 *       Example:
 *
 *          N = 3
 *          W = 3
 *          R = 1
 *
 *       Write:
 *
 *          Coordinator ko teeno replicas ke ACK ka wait karna padega.
 *
 *       Read:
 *
 *          Coordinator ko sirf ek replica ka response chahiye.
 *
 *       Result:
 *
 *          Fast Reads
 *          Slower Writes
 *
 *
 *    Optimization 2: Fast Writes
 *
 *       Agar writes ko fast banana hai:
 *
 *          W = 1
 *          R = N
 *
 *       Example:
 *
 *          N = 3
 *          W = 1
 *          R = 3
 *
 *       Write:
 *
 *          Coordinator ko sirf ek ACK chahiye.
 *
 *       Read:
 *
 *          Coordinator ko teeno replicas ke responses ka wait
 *          karna padega.
 *
 *       Result:
 *
 *          Fast Writes
 *          Slower Reads
 *
 *
 *    Complete Quorum Flow:
 *
 *                         Client
 *                            |
 *                            v
 *                     +-------------+
 *                     | Coordinator |
 *                     +------+------+
 *                            |
 *                 +----------+----------+
 *                 |          |          |
 *                 v          v          v
 *               Replica    Replica    Replica
 *                  1          2          3
 *                 |          |          |
 *                 +----------+----------+
 *                            |
 *                            v
 *                    Collect Responses
 *                            |
 *                     +------+------+
 *                     |             |
 *                     v             v
 *                 Write W         Read R
 *                     |             |
 *                     +------+------+
 *                            |
 *                            v
 *                      Quorum Met
 *                            |
 *                            v
 *                     Return Response
 *                            |
 *                            v
 *                          Client
 *
 *
 *    Simple words mein:
 *
 *    Client ki request ek Coordinator receive karta hai.
 *
 *    Coordinator responsible replicas ko request bhejta hai aur
 *    unke responses collect karta hai.
 *
 *    N:
 *       Total replicas.
 *
 *    W:
 *       Write successful hone ke liye Coordinator ko required
 *       acknowledgements.
 *
 *    R:
 *       Read complete hone ke liye Coordinator ko required responses.
 *
 *    Smaller R/W:
 *       -> Lower latency.
 *
 *    Larger R/W:
 *       -> Stronger consistency ke liye useful.
 *
 *    Common quorum overlap condition:
 *
 *         W + R > N
 *
 *    Example:
 *
 *         N = 3
 *         W = 2
 *         R = 2
 *
 *
 *    Core Idea:
 *
 *          Client
 *             |
 *             v
 *        Coordinator
 *             |
 *        +----+----+
 *        |    |    |
 *        v    v    v
 *       S1   S2   S3
 *        \    |    /
 *         \   |   /
 *          Responses
 *             |
 *             v
 *        Quorum Check
 *             |
 *             v
 *          Client
 *
 *    Yaani Coordinator client aur replicas ke beech request ko
 *    coordinate karta hai aur R/W quorum satisfy hone ke baad
 *    client ko final response return karta hai.
 *
 *    Note:
 *    W + R > N quorum overlap provide karta hai, lekin real distributed
 *    systems mein strict/linearizable consistency ke liye versioning,
 *    concurrent writes, failure handling aur implementation details bhi
 *    correctly handle karne padte hain.
 */

In [ ]:
/**
 * 4. Consistency Models:
 *    - Ab hum samajhte hain ki replicated data ko system mein kitna consistent rakhna
 *      hai.
 *    - Maan lo ek Key ki multiple replicas hain: Key: user:123
 * 
 *              +---------> Server 1 -> Value A
 *              |
 *         Key -+---------> Server 2 -> Value A
 *              |
 *              +---------> Server 3 -> Value A
 * 
 *    - Ab client ek new value write karta hai: put(user:123, B)
 *
 *    Ab question hai:
 *    - "Kya next read mein har server se immediately B milega, ya kuch servers 
 *       temporarily A return kar sakte hain?"
 *    - Isi behavior ko Consistency Model define karta hai.
*/

/**
 * a. Strong Consistency
 *    - Strong Consistency mein REAAD hamesha latest successful WRITE ki value return
 *      karta hai.
 * 
 *    - Write: put(user:123, B)
 *                  |
 *                  v
 *           Latest Value = B
 * 
 *    - Next Read: get(user:123)
 *                      |
 *                      v
 *                      B
 *
 *    - Matlab ek baar write successfully complete ho gaya, uske baad read ko 
 *      old value A return nahi karni chahiye.
 *
 * Simple words mein:
 *
 *         Write B
 *           |
 *           v
 *         Read
 *           |
 *           v
 *         Always B
 *
 * - Strong consistency ka benefit hai correctness.
 * - Lekin iske liye system ko replicas ke synchronization ka wait karna pad sakta hai.
 * - Agar required replica available nahi hai:
 *
 *         Client
 *            |
 *            v
 *       Coordinator
 *            |
 *       +----+----+----+
 *       |    |    |    |
 *       v    v    v    v
 *      S1   S2   S3   S4
 *       |    |    X    |
 *
 * - System operation ko wait ya reject kar sakta hai taaki latest data guarantee 
 *   maintain rahe.
 * - Is wajah se Strong Consistency availability aur latency ko negatively affect kar
 *   sakti hai, especially failures ke time.
*/

/**
 *
 *    b. Weak Consistency
 *
 *    Weak Consistency mein read ko latest value milna guaranteed nahi hai.
 *
 *    Example:
 *
 *         Write:
 *
 *         put(user:123, B)
 *
 *         Kuch replicas:
 *
 *         Server 1 -> B
 *         Server 2 -> B
 *         Server 3 -> A
 *
 *         Ab read Server 3 se aati hai:
 *
 *         get(user:123)
 *                |
 *                v
 *                A
 *
 *    Read ko stale/old data mil sakta hai.
 *
 *    Simple words mein:
 *
 *         Latest data immediately milna guaranteed nahi hai.
 *
 *    Iska benefit ye hai ki system latency aur availability
 *    ko prioritize kar sakta hai.
 *
 *
 *    c. Eventual Consistency
 *
 *    Eventual Consistency mein replicas immediately synchronized
 *    hona zaroori nahi hai.
 *
 *    Lekin agar new updates stop ho jaayein aur system ko enough
 *    time diya jaaye, to replicas eventually same/latest value
 *    par converge kar jaayengi.
 *
 *
 *         Initial:
 *
 *         S1 -> A
 *         S2 -> A
 *         S3 -> A
 *
 *              |
 *              | Write B
 *              v
 *
 *         Temporary State:
 *
 *         S1 -> B
 *         S2 -> B
 *         S3 -> A
 *
 *              |
 *              | Replication / Synchronization
 *              v
 *
 *         Final State:
 *
 *         S1 -> B
 *         S2 -> B
 *         S3 -> B
 *
 *    Yaani temporary inconsistency allowed hai,
 *    lekin eventually replicas converge kar jaati hain.
 *
 *
 *    d. Strong Consistency vs Availability
 *
 *    Strong Consistency ko maintain karne ke liye system ko
 *    replica responses ka wait karna pad sakta hai.
 *
 *         Strong Consistency
 *                |
 *                v
 *         Wait for required state
 *                |
 *                v
 *         Higher correctness
 *                |
 *                v
 *         Potentially lower availability
 *
 *    Agar required replicas unavailable hain:
 *
 *         Request
 *            |
 *            v
 *       Coordinator
 *            |
 *            X
 *       Required Replica
 *        unavailable
 *
 *    System latest-data guarantee maintain karne ke liye
 *    operation ko block/reject kar sakta hai.
 *
 *    Isliye Strong Consistency aur High Availability ke beech
 *    trade-off aa sakta hai.
 *
 *
 *    Eventual Consistency mein system temporary stale data allow karta hai.
 *
 *         Request
 *            |
 *            v
 *       Available Replica
 *            |
 *            v
 *       Return Data
 *
 *    Isse system failures ke time bhi requests serve kar sakta hai,
 *    lekin returned value temporarily stale ho sakti hai.
 *
 *
 *    e. Dynamo and Cassandra
 *
 *    Dynamo-style systems aur Cassandra jaise distributed databases
 *    eventual consistency ko support kar sakte hain.
 *
 *    In systems ka design high availability aur scalability ko
 *    strongly prioritize karta hai.
 *
 *    Matlab replicas ke beech temporary inconsistency acceptable
 *    ho sakti hai, as long as replicas eventually converge kar jaayein.
 *
 *
 *    f. Temporary Conflicts and Versioning
 *
 *    Eventual Consistency mein ek important problem hai:
 *
 *         "Agar different replicas par temporarily different values hain,
 *          to final correct value kaise decide karenge?"
 *
 *    Example:
 *
 *         Server 1 -> Value B
 *         Server 2 -> Value C
 *
 *    Ab conflict hai.
 *
 *    Is conflict ko resolve karne ke liye versioning jaise mechanisms
 *    use kiye ja sakte hain.
 *
 *         Value B -> Version 10
 *         Value C -> Version 11
 *
 *    System version information ko compare karke decide kar sakta hai
 *    ki kaunsi value newer hai.
 *
 *    Distributed systems mein timestamps, logical clocks,
 *    vector clocks ya other versioning mechanisms use case ke
 *    according conflict detection/resolution mein help kar sakte hain.
 *
 *
 *    Simple words mein:
 *
 *    Strong Consistency:
 *
 *       -> Read always latest successful write return karta hai.
 *       -> Correctness strong hoti hai.
 *       -> Failure ke time operations wait/block ho sakti hain.
 *
 *
 *    Weak Consistency:
 *
 *       -> Read stale data return kar sakti hai.
 *       -> Latest value immediately guaranteed nahi hai.
 *       -> Availability/latency better ho sakti hai.
 *
 *
 *    Eventual Consistency:
 *
 *       -> Replicas temporarily different ho sakti hain.
 *       -> Time ke saath replicas same/latest state par converge karengi.
 *       -> Temporary conflicts possible hain.
 *       -> Versioning/conflict-resolution mechanisms use kiye ja sakte hain.
 *
 *
 *    Core Idea:
 *
 *         Strong Consistency
 *              |
 *              v
 *         Latest data immediately
 *              |
 *              v
 *         More coordination
 *
 *
 *         Eventual Consistency
 *              |
 *              v
 *         Temporary stale data allowed
 *              |
 *              v
 *         Less coordination
 *              |
 *              v
 *         Higher availability
 *
 *
 *    Yaani Consistency Model basically ye decide karta hai:
 *
 *    "Distributed replicas ke across data kitni jaldi aur
 *     kis guarantee ke saath consistent hona chahiye?"
 */


In [ ]:
/**
 * 5. Vector Clocks:
 *
 *    Ab hum samajhte hain ki distributed Key-Value Store mein
 *    different versions of data ko kaise track karenge.
 *
 *    Problem ye hai ki distributed system mein same data ki
 *    multiple replicas ho sakti hain aur different servers par
 *    updates same time ke aas-paas aa sakte hain.
 *
 *    Example:
 *
 *         Server 1 -> Value B
 *         Server 2 -> Value C
 *
 *    Ab question hai:
 *
 *    "Kaunsi value newer hai?"
 *
 *    Simple timestamp hamesha enough nahi hota, especially jab
 *    different servers independently updates kar rahe hon.
 *
 *    Is problem ko handle karne ke liye Vector Clocks use kiye
 *    ja sakte hain.
 *
 *
 *    Vector Clock:
 *
 *    Vector Clock data ke version ko track karta hai using
 *    (server, version) pairs.
 *
 *         Server + Version
 *
 *    Example:
 *
 *         D([S1, 2], [S2, 1], [S3, 3])
 *
 *    Iska matlab:
 *
 *         S1 -> version 2
 *         S2 -> version 1
 *         S3 -> version 3
 *
 *    Yaani vector clock batata hai ki different servers ke
 *    according data ka version state kya hai.
 *
 *
 *    Basic Representation:
 *
 *         D([S1, v1], [S2, v2], [Sn, vn])
 *
 *    Yahan:
 *
 *         S1, S2, Sn -> Servers
 *         v1, v2, vn -> Un servers ke corresponding versions
 *
 *
 *    Example:
 *
 *         D([S1, 2], [S2, 1], [S3, 4])
 *
 *         S1 -> 2
 *         S2 -> 1
 *         S3 -> 4
 *
 *
 *    Ab maan lo initially data Server S1 par write hota hai.
 *
 *         Initial:
 *
 *         D()
 *
 *    S1 par first write:
 *
 *         S1 exists nahi karta.
 *
 *         Isliye:
 *
 *         D([S1, 1])
 *
 *
 *    Ab S1 par second write hota hai.
 *
 *         Existing entry:
 *
 *         [S1, 1]
 *
 *         S1 ka version increment hoga:
 *
 *         [S1, 2]
 *
 *         Result:
 *
 *         D([S1, 2])
 *
 *
 *    Ab S2 bhi same data par write karta hai.
 *
 *         Current vector:
 *
 *         D([S1, 2])
 *
 *    S2 vector mein exist nahi karta.
 *
 *         Isliye new entry add hogi:
 *
 *         D([S1, 2], [S2, 1])
 *
 *
 *    Rule simple hai:
 *
 *         Agar Server Si ka entry already exist karta hai:
 *
 *             vi = vi + 1
 *
 *         Agar Server Si ka entry exist nahi karta:
 *
 *             Add [Si, 1]
 *
 *
 *    Example:
 *
 *         Current:
 *
 *         D([S1, 2], [S2, 1])
 *
 *         Write at S1:
 *
 *         D([S1, 3], [S2, 1])
 *
 *
 *         Write at S3:
 *
 *         D([S1, 3], [S2, 1], [S3, 1])
 *
 *
 *    Vector Clock ka main purpose hai versions ke beech
 *    ordering aur conflicts detect karna.
 *
 *
 *    Example:
 *
 *         Version A:
 *
 *         D([S1, 2])
 *
 *         Version B:
 *
 *         D([S1, 3])
 *
 *    Yahan clearly B newer hai because S1 ka version:
 *
 *         3 > 2
 *
 *
 *    Lekin distributed system mein ek interesting case aata hai.
 *
 *         Version A:
 *
 *         D([S1, 2])
 *
 *         Version B:
 *
 *         D([S1, 1], [S2, 1])
 *
 *    Dono vectors mein ek simple "newer" relationship
 *    necessarily establish nahi hota.
 *
 *    Ye indicate kar sakta hai ki different servers ne
 *    independently updates kiye hain.
 *
 *    Yaani possible conflict:
 *
 *         S1 -> Update A
 *         S2 -> Update B
 *              |
 *              v
 *           Conflict
 *
 *    Vector Clock aise concurrent/conflicting versions ko
 *    detect karne mein help karta hai.
 *
 *
 *    Complete Idea:
 *
 *         Client
 *            |
 *            v
 *       Distributed Store
 *            |
 *       +----+----+
 *       |         |
 *       v         v
 *      S1         S2
 *       |         |
 *       v         v
 *    Version A  Version B
 *       |         |
 *       +----+----+
 *            |
 *            v
 *      Vector Clocks
 *            |
 *            v
 *    Ordering / Conflict Detection
 *
 *
 *    Simple words mein:
 *
 *    Vector Clock ek version-tracking mechanism hai jo batata hai
 *    ki different servers ne data par kitne updates kiye hain.
 *
 *    Har server ke liye ek version number maintain kiya jaata hai.
 *
 *         D([S1, v1], [S2, v2], [Sn, vn])
 *
 *    Jab Server Si par write hota hai:
 *
 *         -> Agar Si already present hai:
 *               version increment karo.
 *
 *         -> Agar Si present nahi hai:
 *               [Si, 1] add karo.
 *
 *    Is information ka use distributed system mein versions ko
 *    compare karne aur concurrent updates/conflicts ko detect
 *    karne ke liye kiya ja sakta hai.
 *
 *
 *    Core Idea:
 *
 *         Write at S1
 *              |
 *              v
 *         Increment S1 version
 *              |
 *              v
 *         D([S1, v1])
 *
 *
 *         Write at S2
 *              |
 *              v
 *         Add / Increment S2
 *              |
 *              v
 *         D([S1, v1], [S2, v2])
 *
 *    Yaani Vector Clock distributed Key-Value Store ko ye
 *    samajhne mein help karta hai ki different versions ke
 *    beech ordering hai ya conflict.
 */

In [ ]:
/**
 * 5. Vector Clocks:
 *
 *    Vector Clock ek mechanism hai jo distributed system mein
 *    different versions of the same data ko track karta hai.
 *
 *    Har server ke liye ek version number maintain kiya jaata hai.
 *
 *    Example:
 *
 *       D([S1, v1], [S2, v2], [Sn, vn])
 *
 *    Yahan:
 *
 *       S1, S2, Sn -> Servers
 *       v1, v2, vn -> Un servers ke version numbers
 *
 *    Jab Server Si par write hota hai:
 *
 *       -> Agar Si ka entry already exist karta hai:
 *             vi ko increment karo.
 *
 *       -> Agar Si ka entry exist nahi karta:
 *             [Si, 1] add karo.
 *
 *
 *    Example:
 *
 *       Initial:
 *          D()
 *
 *       Write at S0:
 *          D([S0, 1])
 *
 *       Another write at S0:
 *          D([S0, 2])
 *
 *       Write at S1:
 *          D([S0, 2], [S1, 1])
 *
 *
 *    Vector Clock ka main purpose hai:
 *
 *       -> Versions ke beech ordering detect karna.
 *       -> Concurrent updates ko identify karna.
 *       -> Data conflicts detect karna.
 *
 *
 *    Vector Clock: Conflict Detection
 *
 *    Ab maan lo same data ki do versions X aur Y hain.
 *
 *    Humein determine karna hai:
 *
 *       "Kya Y, X ka newer version hai?"
 *
 *       Ya:
 *
 *       "Kya X aur Y independently modify hue hain aur
 *        unke beech conflict hai?"
 *
 *
 *    1. Ancestor / No Conflict
 *
 *    Agar Version Y ke har counter ki value Version X se
 *    greater than or equal hai, to Y ko X ka newer version
 *    maana ja sakta hai.
 *
 *       Version X:
 *
 *          [S0:1, S1:1]
 *
 *       Version Y:
 *
 *          [S0:1, S1:2]
 *
 *       Compare:
 *
 *          S0: 1 >= 1  -> Yes
 *          S1: 2 >= 1  -> Yes
 *
 *       Isliye:
 *
 *          X -> Y
 *
 *       Y, X ka descendant/newer version hai.
 *
 *       Is case mein conflict nahi hai.
 *
 *
 *       Visual:
 *
 *          X
 *          |
 *          | Update at S1
 *          v
 *          Y
 *
 *
 *    2. Sibling / Conflict
 *
 *    Ab ek different situation dekho:
 *
 *       Version X:
 *
 *          [S0:1, S1:2]
 *
 *       Version Y:
 *
 *          [S0:2, S1:1]
 *
 *       Compare:
 *
 *          S0:
 *             Y = 2 > X = 1
 *
 *          S1:
 *             Y = 1 < X = 2
 *
 *       Y har counter mein X se greater/equal nahi hai.
 *
 *       Similarly X bhi har counter mein Y se
 *       greater/equal nahi hai.
 *
 *       Isliye dono versions ko directly order nahi kiya ja sakta.
 *
 *              X
 *             / \
 *            /   \
 *       Update   Update
 *          /       \
 *         v         v
 *        X1         Y1
 *
 *       Ye concurrent updates ko indicate karta hai.
 *
 *       In versions ko Siblings kaha ja sakta hai.
 *
 *       Matlab:
 *
 *          X vs Y
 *             |
 *             v
 *          Conflict
 *
 *
 *    Simple comparison:
 *
 *       Ancestor:
 *
 *          [S0:1, S1:1]
 *                    |
 *                    v
 *          [S0:1, S1:2]
 *
 *          One version is newer than the other.
 *          No conflict.
 *
 *
 *       Sibling:
 *
 *          [S0:1, S1:2]
 *                    X
 *
 *                    vs
 *
 *          [S0:2, S1:1]
 *                    Y
 *
 *          Neither version dominates the other.
 *          Conflict detected.
 *
 *
 *    3. Conflict Resolution
 *
 *    Agar siblings milte hain, to system ko decide karna padega
 *    ki final value kya honi chahiye.
 *
 *       Version X
 *          |
 *          +------+
 *                 |
 *              Conflict
 *                 |
 *          +------+
 *          |
 *       Version Y
 *
 *    Kuch systems mein client ko conflicting versions return
 *    ki ja sakti hain aur client decide karta hai ki final value
 *    kya honi chahiye.
 *
 *       Server
 *          |
 *          v
 *       Conflicting Versions
 *          |
 *          v
 *       Client
 *          |
 *          v
 *       Resolve Conflict
 *
 *    Is approach ka drawback ye hai ki client ko
 *    conflict-resolution logic handle karna padta hai.
 *
 *
 *    4. Drawbacks of Vector Clocks
 *
 *    Vector Clocks powerful hain, lekin kuch limitations bhi hain.
 *
 *
 *    Drawback 1: Client-side Conflict Resolution
 *
 *       Agar multiple concurrent versions create ho jaayein,
 *       to client ko conflict resolve karna pad sakta hai.
 *
 *          Server
 *             |
 *             v
 *          Siblings
 *             |
 *             v
 *          Client
 *             |
 *             v
 *       Resolve Conflict
 *
 *       Isse application complexity increase ho sakti hai.
 *
 *
 *    Drawback 2: Vector Clock Size
 *
 *       Agar system mein bahut saare servers participate karte hain,
 *       to vector clock mein bahut saare server-version entries
 *       accumulate ho sakte hain.
 *
 *       Example:
 *
 *          D(
 *            [S1:10],
 *            [S2:5],
 *            [S3:8],
 *            [S4:2],
 *            [S5:7],
 *            ...
 *            [Sn:4]
 *          )
 *
 *       Large vector clock ka matlab:
 *
 *          -> More storage
 *          -> More network overhead
 *          -> More comparison cost
 *
 *
 *    5. Mitigation: Limit Vector Clock Size
 *
 *    Vector Clock ko indefinitely grow nahi hone dete.
 *
 *    Clock size ko limit karne ke liye oldest entries ko remove
 *    kiya ja sakta hai.
 *
 *       Large Clock:
 *
 *          [S1:10]
 *          [S2:8]
 *          [S3:7]
 *          [S4:6]
 *          [S5:5]
 *          [S6:4]
 *
 *                  |
 *                  | Limit Size
 *                  v
 *
 *       Smaller Clock:
 *
 *          [S1:10]
 *          [S2:8]
 *          [S3:7]
 *
 *    Isse vector clock ka size manageable rehta hai.
 *
 *    Kuch information lose ho sakti hai, lekin practical distributed
 *    systems mein clock size ko bounded rakhna acceptable trade-off
 *    ho sakta hai.
 *
 *    Dynamo-style systems mein vector-clock size ko limit karne
 *    jaise approaches use kiye gaye hain.
 *
 *
 *    Simple words mein:
 *
 *    Vector Clock har server ke updates ko track karta hai.
 *
 *       D([S0:1], [S1:2])
 *
 *    Agar Version Y ke saare counters Version X se
 *    greater/equal hain:
 *
 *       X -> Y
 *
 *    to Y newer version hai aur conflict nahi hai.
 *
 *    Lekin agar:
 *
 *       X = [S0:1, S1:2]
 *       Y = [S0:2, S1:1]
 *
 *    to dono versions incomparable hain.
 *    Ye concurrent updates ko indicate karta hai aur conflict
 *    detect hota hai.
 *
 *    Conflict hone par client/application ko versions resolve
 *    karne ki zarurat pad sakti hai.
 *
 *    Vector Clock ka size bhi bahut bada ho sakta hai,
 *    isliye practical systems mein oldest entries ko remove karke
 *    clock size ko limit kiya ja sakta hai.
 *
 *
 *    Core Idea:
 *
 *       Write
 *         |
 *         v
 *    Update Vector Clock
 *         |
 *         v
 *    Compare Versions
 *         |
 *       +---+---+
 *       |       |
 *    Ordered  Incomparable
 *       |       |
 *       v       v
 *    No Conflict  Conflict
 *                   |
 *                   v
 *             Resolve Versions
 */

In [ ]:
/**
 * 6. Failure Detection:
 *
 *    Distributed Key-Value Store mein multiple servers hote hain.
 *    Ab ek important question hai:
 *
 *    "Agar koi server fail ho gaya, to baaki servers ko kaise pata chalega?"
 *
 *    Isi problem ko Failure Detection kehte hain.
 *
 *
 *    Example:
 *
 *         +---------+     +---------+     +---------+
 *         | Server 1|     | Server 2|     | Server 3|
 *         +---------+     +---------+     +---------+
 *              |               |               |
 *              |               |               |
 *              +---------------+---------------+
 *
 *    Maan lo Server 2 suddenly unavailable ho gaya.
 *
 *         Server 1  ----X----> Server 2
 *         Server 3  ----X----> Server 2
 *
 *    Ab system ko determine karna hai:
 *
 *         "Kya Server 2 actually fail ho gaya hai?"
 *
 *
 *    Problem 1: Single Node ki Opinion Sufficient Nahi Hai
 *
 *    Maan lo Server 1 ne Server 2 ko request bheji,
 *    lekin Server 2 se response nahi aaya.
 *
 *         Server 1
 *             |
 *             | Ping
 *             X
 *         Server 2
 *
 *    Server 1 ye assume kar sakta hai:
 *
 *         "Server 2 failed."
 *
 *    Lekin zaroori nahi ki Server 2 actually fail ho gaya ho.
 *
 *    Possible reasons:
 *
 *         -> Network temporarily slow hai.
 *         -> Network connection break hai.
 *         -> Server overloaded hai.
 *         -> Server response dene mein late hai.
 *
 *    Matlab:
 *
 *         No Response
 *             |
 *             +--> Server Failed?
 *             |
 *             +--> Network Problem?
 *             |
 *             +--> Temporary Delay?
 *
 *    Isliye ek single server ki opinion ke basis par
 *    kisi node ko definitely failed declare karna unsafe hai.
 *
 *
 *    Problem 2: All-to-All Multicasting
 *
 *    Ek simple idea ho sakta hai ki har server baaki
 *    har server ko continuously check kare.
 *
 *         Server 1 <------> Server 2
 *            ^  \            /  ^
 *            |   \          /   |
 *            |    \        /    |
 *            v     \      /     v
 *         Server 3 <------> Server 4
 *
 *    Agar N servers hain, to bahut saare health-check messages
 *    generate honge.
 *
 *         More Servers
 *              |
 *              v
 *       More Health Checks
 *              |
 *              v
 *       More Network Messages
 *              |
 *              v
 *       Higher Overhead
 *
 *    Isliye large distributed systems mein all-to-all multicasting
 *    inefficient ho sakta hai.
 *
 *
 *    Solution: Gossip Protocol
 *
 *    All-to-all communication ke instead hum Gossip Protocol use
 *    kar sakte hain.
 *
 *    Gossip Protocol mein har node apni membership list maintain
 *    karta hai.
 *
 *    Membership List mein mainly:
 *
 *         Node ID + Heartbeat
 *
 *    Example:
 *
 *         Server 1 Membership List:
 *
 *         +------+-----------+
 *         | Node | Heartbeat |
 *         +------+-----------+
 *         | S1   |    10     |
 *         | S2   |    25     |
 *         | S3   |    17     |
 *         | S4   |    31     |
 *         +------+-----------+
 *
 *
 *    Har node periodically apna heartbeat increment karta hai.
 *
 *         S1:
 *
 *         Heartbeat 10
 *              |
 *              v
 *         Heartbeat 11
 *              |
 *              v
 *         Heartbeat 12
 *              |
 *              v
 *         Heartbeat 13
 *
 *    Ye heartbeat signal karta hai ki node alive hai.
 *
 *
 *    Ab node apna updated heartbeat random peers ko gossip karta hai.
 *
 *         S1
 *        /  \
 *       v    v
 *      S2   S4
 *
 *    S1 apni information share karta hai:
 *
 *         "S1 heartbeat = 13"
 *
 *    S2 aur S4 is information ko apni membership list mein
 *    update kar lete hain.
 *
 *    Similarly doosre nodes bhi apne heartbeats ko
 *    random peers ke saath share karte hain.
 *
 *
 *    Example:
 *
 *         S1 ---> S3
 *         S3 ---> S4
 *         S4 ---> S2
 *
 *    Is tarah information gradually cluster ke multiple
 *    nodes tak spread hoti rehti hai.
 *
 *
 *    Membership Lists continuously update hoti rehti hain:
 *
 *         Receive Gossip
 *              |
 *              v
 *       Compare Heartbeat
 *              |
 *              v
 *       Update Membership List
 *
 *
 *    Is approach mein har node ko har doosre node se
 *    directly communicate karne ki zarurat nahi hoti.
 *
 *    Isliye large-scale systems mein communication overhead
 *    all-to-all multicasting ke comparison mein significantly
 *    better manage kiya ja sakta hai.
 *
 *
 *    Failure Rule:
 *
 *    Ab question hai:
 *
 *         "Kaise decide karenge ki koi node fail ho gaya?"
 *
 *    Maan lo Server S2 ka heartbeat normally increase ho raha tha:
 *
 *         20 -> 21 -> 22 -> 23 -> 24
 *
 *    Lekin suddenly heartbeat update hona stop ho gaya:
 *
 *         24 -> 24 -> 24 -> 24
 *
 *    Agar defined timeout period ke andar S2 ka heartbeat
 *    increase nahi hota, to S2 ko potentially DOWN/FAILED
 *    suspect kiya ja sakta hai.
 *
 *
 *         Last Heartbeat
 *                |
 *                v
 *             Timeout
 *                |
 *                v
 *        No Heartbeat Update
 *                |
 *                v
 *         Node Suspected Down
 *
 *
 *    Lekin sirf ek node ko heartbeat update nahi mila,
 *    to immediately permanent failure declare karna risky hai.
 *
 *    Ho sakta hai:
 *
 *         -> Network issue ho.
 *         -> Gossip message lost ho gaya ho.
 *         -> Temporary network partition ho.
 *
 *    Isliye failure ko multiple independent observations se
 *    confirm karna safer hai.
 *
 *
 *         S1 -> S2 heartbeat missing
 *         S3 -> S2 heartbeat missing
 *         S4 -> S2 heartbeat missing
 *                    |
 *                    v
 *          Multiple Observations
 *                    |
 *                    v
 *            Strong Failure Suspicion
 *                    |
 *                    v
 *             Mark S2 as Down
 *
 *
 *    Yaani:
 *
 *         Single Observation
 *                |
 *                v
 *          Failure Suspicion
 *
 *         Multiple Independent Observations
 *                |
 *                v
 *          Stronger Confirmation
 *
 *
 *    Complete Failure Detection Flow:
 *
 *         Every Node
 *             |
 *             v
 *      Increment Own Heartbeat
 *             |
 *             v
 *      Select Random Peers
 *             |
 *             v
 *       Gossip Heartbeat
 *             |
 *             v
 *      Update Membership List
 *             |
 *             v
 *      Monitor Other Heartbeats
 *             |
 *             v
 *      Heartbeat Timeout?
 *          /          \
 *        No            Yes
 *        |              |
 *        v              v
 *      Alive         Suspect Node
 *                         |
 *                         v
 *                Multiple Observations
 *                         |
 *                   +-----+-----+
 *                   |           |
 *               Confirmed    Not Confirmed
 *                   |           |
 *                   v           v
 *               Mark Down   Keep Monitoring
 *
 *
 *    Simple words mein:
 *
 *    Failure Detection ka goal hai:
 *
 *         "Reliable way se identify karna ki kaunsa server
 *          available hai aur kaunsa potentially failed hai."
 *
 *    All-to-all multicasting large scale par expensive ho sakti hai,
 *    isliye Gossip Protocol use kiya ja sakta hai.
 *
 *    Gossip Protocol mein:
 *
 *         -> Har node membership list maintain karta hai.
 *         -> Har node apna heartbeat periodically increment karta hai.
 *         -> Heartbeat random peers ko gossip kiya jaata hai.
 *         -> Nodes received heartbeat information se apni
 *            membership lists update karte hain.
 *
 *    Failure Rule:
 *
 *         -> Agar kisi node ka heartbeat defined timeout ke andar
 *            increase nahi hota, to node ko suspect kiya jaata hai.
 *         -> Multiple independent observations failure ko
 *            confirm karne mein help karti hain.
 *
 *
 *    Core Idea:
 *
 *         Distributed Servers
 *                |
 *                v
 *         Failure Detection
 *                |
 *         +------+------+ 
 *         |             |
 *    Single Opinion   All-to-All
 *         |             |
 *         X             X
 *    Insufficient     Expensive
 *                |
 *                v
 *         Gossip Protocol
 *                |
 *                v
 *       Heartbeat Exchange
 *                |
 *                v
 *       Membership Lists
 *                |
 *                v
 *         Timeout Check
 *                |
 *                v
 *       Multiple Observations
 *                |
 *                v
 *         Failure Detection
 */


In [ ]:
/**
 * 7. Handling Permanent Failures:
 *
 *    Ab humne failure detection samajh liya.
 *    Lekin failure detect karna sirf first step hai.
 *
 *    Ab question hai:
 *
 *    "Agar koi node permanently fail ho gaya aur kabhi wapas nahi aaya,
 *     to us node par stored data ka kya hoga?"
 *
 *
 *    Distributed Key-Value Store mein same data ki multiple replicas
 *    different servers par hoti hain.
 *
 *         Key: user:123
 *              |
 *        +-----+-----+-----+
 *        |           |     |
 *        v           v     v
 *       S1          S2    S3
 *        |           |     |
 *        A           A     A
 *
 *    Agar S2 permanently fail ho gaya:
 *
 *         S1 -> A
 *         S2 -> X  FAILED
 *         S3 -> A
 *
 *    To remaining replicas ke paas abhi bhi data available hai.
 *
 *
 *    Lekin replicas ke data mein divergence aa sakta hai.
 *
 *    Replica Divergence ka matlab hai ki different replicas ke paas
 *    same key ki different versions stored ho jaayein.
 *
 *
 *    Ye different reasons ki wajah se ho sakta hai:
 *
 *
 *    1. Permanent Node Failure
 *
 *    Maan lo S2 permanently fail ho gaya.
 *
 *         S1 -> Value A
 *         S2 -> Value A  X
 *         S3 -> Value A
 *
 *    Baad mein S1 par new update aata hai:
 *
 *         put(user:123, B)
 *
 *         S1 -> B
 *         S2 -> X  FAILED
 *         S3 -> A
 *
 *    Ab replicas temporarily different state mein hain.
 *
 *
 *    2. Network Partition
 *
 *    Kabhi-kabhi servers alive hote hain, lekin network problem ki
 *    wajah se ek group doosre group se communicate nahi kar pata.
 *
 *         +----------------+       X       +----------------+
 *         | Data Center 1  |---------------| Data Center 2  |
 *         |                |   Network     |                |
 *         | S1     S2      |   Partition   | S3     S4      |
 *         +----------------+               +----------------+
 *
 *    Dono sides independently requests process kar sakti hain.
 *
 *         S1 -> Value B
 *         S3 -> Value C
 *
 *    Network restore hone ke baad:
 *
 *         S1 -> B
 *         S3 -> C
 *
 *    Ab same key ke different versions exist kar sakte hain.
 *
 *
 *    3. Sloppy Quorum / Hinted Handoff
 *
 *    Kabhi primary replica temporarily unavailable hoti hai.
 *
 *    System availability maintain karne ke liye ek temporary node
 *    write accept kar sakta hai.
 *
 *         Original Replica
 *              |
 *              X
 *          Unavailable
 *
 *         Coordinator
 *              |
 *              v
 *       Temporary Node
 *              |
 *              v
 *         Accept Write
 *
 *    Is temporary node ko baad mein original replica ke available
 *    hone par data handoff kiya ja sakta hai.
 *
 *    Is technique ko Hinted Handoff ke context mein use kiya ja sakta hai.
 *
 *
 *    In situations ki wajah se:
 *
 *         Replica 1 -> Version A
 *         Replica 2 -> Version B
 *         Replica 3 -> Version C
 *
 *    Matlab same key ke different replicas ke paas different
 *    versions ho sakte hain.
 *
 *
 *    Ab problem:
 *
 *    "Agar hum synchronization na karein to kya hoga?"
 *
 *    Different clients ko different/stale values mil sakti hain.
 *
 *         Client 1 ---> S1 ---> Value B
 *
 *         Client 2 ---> S3 ---> Value A
 *
 *    Same key ke liye:
 *
 *         Client 1 -> B
 *         Client 2 -> A
 *
 *    Ye incorrect ya stale behavior create kar sakta hai.
 *
 *
 *    Isliye humein replicas ko eventually synchronize karna hota hai.
 *
 *
 *    Synchronization ka goal:
 *
 *         Different Versions
 *                |
 *                v
 *         Conflict Detection
 *                |
 *                v
 *         Conflict Resolution
 *                |
 *                v
 *         Replica Synchronization
 *                |
 *                v
 *         Consistent State
 *
 *
 *    Example:
 *
 *         Before Synchronization:
 *
 *         S1 -> Version B
 *         S2 -> Version A
 *         S3 -> Version B
 *
 *                  |
 *                  | Synchronization
 *                  v
 *
 *         After Synchronization:
 *
 *         S1 -> Version B
 *         S2 -> Version B
 *         S3 -> Version B
 *
 *
 *    Is process ko eventually consistency maintain karne ke liye
 *    use kiya ja sakta hai.
 *
 *    Yaani replicas immediately consistent hona zaroori nahi,
 *    lekin system ka goal hota hai ki time ke saath replicas
 *    same correct/latest state par converge kar jaayein.
 *
 *
 *    Is synchronization ke liye system:
 *
 *       -> Version information use kar sakta hai.
 *       -> Vector Clocks se conflicts detect kar sakta hai.
 *       -> Conflict resolution perform kar sakta hai.
 *       -> Missing updates ko replicas ke beech synchronize kar sakta hai.
 *
 *
 *    Complete Flow:
 *
 *         Replica Divergence
 *                |
 *        +-------+-------+
 *        |       |       |
 *      Node   Network   Sloppy
 *     Failure  Partition  Quorum
 *        |       |       |
 *        +-------+-------+
 *                |
 *                v
 *       Different Versions
 *                |
 *                v
 *       Conflict Detection
 *                |
 *                v
 *       Conflict Resolution
 *                |
 *                v
 *        Synchronization
 *                |
 *                v
 *       Replicas Converge
 *                |
 *                v
 *       Eventual Consistency
 *
 *
 *    Simple words mein:
 *
 *    Distributed system mein permanent node failures, network
 *    partitions aur sloppy quorum/hinted handoff ki wajah se
 *    replicas temporarily different data store kar sakti hain.
 *
 *    Agar hum synchronization na karein, to different clients ko
 *    stale ya incorrect versions mil sakti hain.
 *
 *    Isliye system ko replicas ke beech missing updates aur
 *    conflicting versions ko detect aur resolve karke eventually
 *    synchronize karna padta hai.
 *
 *    Goal ye hai:
 *
 *         Different Replica States
 *                |
 *                v
 *         Synchronization
 *                |
 *                v
 *       Consistent Replica State
 *
 *    Isi approach ki wajah se system eventual consistency achieve
 *    kar sakta hai.
 *
 *
 *    Core Idea:
 *
 *         Failure / Partition
 *                |
 *                v
 *        Replica Divergence
 *                |
 *                v
 *       Detect + Resolve Conflict
 *                |
 *                v
 *        Synchronize Replicas
 *                |
 *                v
 *       Eventually Consistent
 */

In [ ]:
/**
 * 8. Handling Temporary Failures:
 *
 *    Ab hum permanent failure ke baad temporary failures ko samajhte hain.
 *
 *    Temporary failure ka matlab hai ki node abhi unavailable hai,
 *    lekin future mein recover hokar wapas aa sakta hai.
 *
 *    Example:
 *
 *         S1 -> Healthy
 *         S2 -> Temporarily Down
 *         S3 -> Healthy
 *
 *         S2 X
 *
 *    Aise case mein hum system ko completely stop nahi karna chahte.
 *    Humein available nodes ke through requests serve karte rehna hai.
 *
 *
 *    Problem with Strict Quorum:
 *
 *    Maan lo:
 *
 *         N = 3
 *         W = 2
 *         R = 2
 *
 *    Aur S2 temporarily down ho gaya.
 *
 *         S1 -> Healthy
 *         S2 -> DOWN
 *         S3 -> Healthy
 *
 *    Agar strict quorum use ho raha hai, to system ko required
 *    replicas ka response/acknowledgement wait karna pad sakta hai.
 *
 *         Write
 *           |
 *           v
 *       Coordinator
 *           |
 *       +---+---+---+
 *       |   |   |   |
 *       v   v   v   v
 *      S1  S2  S3
 *       |   X   |
 *       | Down  |
 *       +-------+
 *
 *    Is situation mein strict quorum operation ko block ya fail
 *    kar sakta hai.
 *
 *         Result:
 *
 *         Lower Availability
 *
 *
 *    Solution: Sloppy Quorum
 *
 *    Sloppy Quorum ka main goal hai:
 *
 *         "Agar original replica temporarily unavailable hai,
 *          to available healthy nodes ka use karke request
 *          serve karo."
 *
 *    Isse system failures ke time bhi available reh sakta hai.
 *
 *
 *    Sloppy Quorum mein:
 *
 *       -> Writes: First W healthy nodes ko write kiya jaata hai.
 *       -> Reads : First R healthy nodes se read kiya jaata hai.
 *
 *    Yaani failed/offline nodes ko temporarily skip kiya ja sakta hai.
 *
 *
 *    Example:
 *
 *         Hash Ring:
 *
 *              S1
 *               |
 *          +----+----+
 *         /           \
 *       S4             S2
 *         \           /
 *          +----+----+
 *               |
 *              S3
 *
 *    Maan lo S2 down hai:
 *
 *              S1
 *               |
 *          +----+----+
 *         /           \
 *       S4             S2 X
 *         \           /
 *          +----+----+
 *               |
 *              S3
 *
 *    System S2 ko skip karke next healthy nodes ko use kar sakta hai.
 *
 *
 *    Write Example:
 *
 *       N = 3
 *       W = 2
 *
 *       S2 = DOWN
 *
 *       Coordinator
 *            |
 *            +--> S1 -> Healthy -> Write
 *            |
 *            +--> S2 -> DOWN -> Skip
 *            |
 *            +--> S3 -> Healthy -> Write
 *
 *       W = 2 healthy acknowledgements mil gaye.
 *
 *       Write successful.
 *
 *
 *    Read Example:
 *
 *       N = 3
 *       R = 2
 *
 *       S2 = DOWN
 *
 *       Coordinator
 *            |
 *            +--> S1 -> Healthy -> Read
 *            |
 *            +--> S2 -> DOWN -> Skip
 *            |
 *            +--> S3 -> Healthy -> Read
 *
 *       R = 2 healthy responses mil gaye.
 *
 *       Read successful.
 *
 *
 *    Yaani:
 *
 *       Strict Quorum:
 *
 *          Original replicas unavailable
 *                    |
 *                    v
 *                 Block
 *
 *
 *       Sloppy Quorum:
 *
 *          Original replica unavailable
 *                    |
 *                    v
 *              Skip failed node
 *                    |
 *                    v
 *          Use healthy nodes
 *                    |
 *                    v
 *                 Continue
 *
 *
 *    Lekin ab ek new problem hai:
 *
 *    "Agar temporary replacement node ne data accept kar liya,
 *     to original node recover hone ke baad data wapas kaise milega?"
 *
 *    Is problem ko Hinted Handoff solve karne mein help karta hai.
 *
 *
 *    Hinted Handoff:
 *
 *    Agar original replica temporarily down hai, to koi doosra
 *    healthy node temporarily us replica ka data store kar sakta hai.
 *
 *         Original Replica
 *              S2
 *              X
 *            Down
 *
 *               |
 *               v
 *
 *         Temporary Node
 *              S3
 *              |
 *              v
 *         Store Update
 *
 *    S3 basically S2 ke behalf par temporary update hold karta hai.
 *
 *
 *    Example:
 *
 *         s2 is down
 *
 *         Client
 *            |
 *            v
 *       Coordinator
 *            |
 *       +----+----+
 *       |         |
 *       v         v
 *      S1        S3
 *                 |
 *                 | Temporary storage
 *                 v
 *             S2's Update
 *
 *
 *    Ab jab S2 recover ho jaata hai:
 *
 *         S2
 *          |
 *          v
 *       Recover
 *          |
 *          v
 *       Contact S3
 *          |
 *          v
 *       Receive Missing Updates
 *          |
 *          v
 *       Update S2
 *
 *    Yaani temporary node S3 apne paas stored updates ko
 *    original node S2 ko handoff kar deta hai.
 *
 *
 *    Complete Hinted Handoff Flow:
 *
 *         S2 Down
 *            |
 *            v
 *      S3 stores update
 *            |
 *            v
 *      S2 Recovers
 *            |
 *            v
 *      S3 -> S2
 *            |
 *            v
 *      Missing Updates
 *            |
 *            v
 *      S2 Synchronized
 *
 *
 *    Example:
 *
 *         Before Failure:
 *
 *         S1 -> Value A
 *         S2 -> Value A
 *         S3 -> Value A
 *
 *         S2 temporarily fails:
 *
 *         S1 -> Value A
 *         S2 -> DOWN
 *         S3 -> Value A
 *
 *         New write:
 *
 *            put(key, B)
 *
 *         S1 -> B
 *         S2 -> DOWN
 *         S3 -> B
 *
 *         S3 temporarily keeps the update that S2 missed.
 *
 *         S2 recovers:
 *
 *         S3 -> Send missing update -> S2
 *
 *         Final:
 *
 *         S1 -> B
 *         S2 -> B
 *         S3 -> B
 *
 *
 *    Is approach se system temporary node failure ke time bhi
 *    requests serve kar sakta hai aur original node ke recover
 *    hone ke baad replicas ko synchronize kar sakta hai.
 *
 *
 *    Complete Flow:
 *
 *         Request
 *            |
 *            v
 *       Coordinator
 *            |
 *            v
 *       Check Hash Ring
 *            |
 *            v
 *       Original Node Healthy?
 *          /          \
 *        Yes           No
 *        |              |
 *        v              v
 *    Use Node       Skip Node
 *                       |
 *                       v
 *                Healthy Replacement
 *                       |
 *                       v
 *                 Sloppy Quorum
 *                       |
 *                       v
 *                Store / Serve Data
 *                       |
 *                       v
 *                Hinted Handoff
 *                       |
 *                       v
 *              Original Node Recovers
 *                       |
 *                       v
 *                Sync Missing Data
 *                       |
 *                       v
 *               Replica Consistent
 *
 *
 *    Simple words mein:
 *
 *    Strict Quorum mein agar required replicas unavailable hain,
 *    to read/write block ho sakti hai.
 *
 *    Sloppy Quorum mein failed nodes ko temporarily skip karke
 *    healthy nodes se operation complete kiya jaata hai.
 *
 *    Hinted Handoff ensure karta hai ki jo updates temporary
 *    replacement node ne accept kiye hain, wo original node ke
 *    recover hone ke baad usko wapas mil jaayein.
 *
 *
 *    Core Idea:
 *
 *         Temporary Failure
 *                |
 *                v
 *         Sloppy Quorum
 *                |
 *                v
 *       Healthy Nodes Handle Request
 *                |
 *                v
 *        Hinted Handoff
 *                |
 *                v
 *       Failed Node Recovers
 *                |
 *                v
 *       Missing Data Sync
 *                |
 *                v
 *       Consistent Replicas
 */

In [ ]:
/**
 * 9. Anti-Entropy Protocol:
 *
 *    Distributed Key-Value Store mein replicas ke paas same data ki
 *    multiple copies hoti hain.
 *
 *    Failure, network partition ya temporary unavailability ki wajah se
 *    replicas ke data mein difference aa sakta hai.
 *
 *         Replica 1 -> Data A
 *         Replica 2 -> Data B
 *
 *    Ab humein replicas ko synchronize karke eventually same state
 *    mein lana hai.
 *
 *
 *    Simple approach:
 *
 *    Hum dono replicas ka poora data compare kar sakte hain.
 *
 *         Replica 1
 *              |
 *              | Compare ALL Data
 *              v
 *         Replica 2
 *
 *    Lekin agar system mein millions of key-value pairs hain,
 *    to poora dataset baar-baar compare karna extremely expensive hoga.
 *
 *         Millions of Keys
 *                |
 *                v
 *          Compare Everything
 *                |
 *                v
 *          Huge Network Traffic
 *                |
 *                v
 *          High CPU + Time
 *
 *    Isliye humein ek efficient way chahiye jisse hum quickly
 *    identify kar saken ki replicas ke data mein difference
 *    exactly kahan hai.
 *
 *
 *    Solution: Anti-Entropy Protocol
 *
 *    Anti-Entropy ek synchronization mechanism hai jo replicas ke
 *    beech periodically data compare karke divergent/mismatched
 *    data ko synchronize karta hai.
 *
 *    Iska goal hai:
 *
 *         Different Replica States
 *                |
 *                v
 *         Detect Differences
 *                |
 *                v
 *         Synchronize Data
 *                |
 *                v
 *         Consistent Replicas
 *
 *
 *    Lekin Anti-Entropy ko efficiently perform karne ke liye
 *    hum Merkle Trees ka use kar sakte hain.
 *
 *
 *    Merkle Tree:
 *
 *    Merkle Tree ek hierarchical hash structure hai jo large
 *    dataset ko efficiently compare karne mein help karta hai.
 *
 *    Basic idea:
 *
 *         Data
 *          |
 *          v
 *      Data Buckets
 *          |
 *          v
 *       Leaf Hashes
 *          |
 *          v
 *    Internal Hashes
 *          |
 *          v
 *       Root Hash
 *
 *
 *    1. Leaves:
 *
 *    Dataset ko multiple buckets/ranges of keys mein divide kiya jaata hai.
 *
 *         Dataset
 *            |
 *       +----+----+----+----+
 *       |    |    |    |    |
 *       v    v    v    v    v
 *      B1   B2   B3   B4   B5
 *
 *    Har bucket ke data ka hash calculate kiya jaata hai.
 *
 *         B1 -> Hash H1
 *         B2 -> Hash H2
 *         B3 -> Hash H3
 *         B4 -> Hash H4
 *
 *    Ye hashes Merkle Tree ke leaf nodes ban jaate hain.
 *
 *
 *    2. Internal Nodes:
 *
 *    Ab leaf hashes ko pair-wise combine karke unke parent hashes
 *    calculate kiye jaate hain.
 *
 *             H1234
 *             /   \
 *          H12     H34
 *          / \     / \
 *        H1  H2  H3  H4
 *
 *    Example:
 *
 *         H12 = Hash(H1 + H2)
 *         H34 = Hash(H3 + H4)
 *
 *         Root = Hash(H12 + H34)
 *
 *
 *    3. Root Hash:
 *
 *    Sabhi lower-level hashes ko combine karne ke baad ek final
 *    Root Hash milta hai.
 *
 *             +-----------+
 *             | Root Hash  |
 *             +-----+-----+
 *                   |
 *             +-----+-----+
 *             |           |
 *             v           v
 *           H12          H34
 *          /  \         /  \
 *         H1  H2        H3  H4
 *         |   |         |   |
 *         B1  B2        B3  B4
 *
 *    Root Hash poore dataset ki summary represent karta hai.
 *
 *
 *    Ab do replicas ko compare karte hain.
 *
 *
 *    Case 1: Same Root Hash
 *
 *         Replica 1:
 *              Root = ABC123
 *
 *         Replica 2:
 *              Root = ABC123
 *
 *    Same root hash ka matlab hai ki dono replicas ka corresponding
 *    data same hai.
 *
 *         Replica 1
 *              |
 *          Root ABC123
 *              |
 *              = 
 *              |
 *          Root ABC123
 *              |
 *         Replica 2
 *
 *         No synchronization required.
 *
 *
 *    Case 2: Different Root Hash
 *
 *         Replica 1:
 *              Root = ABC123
 *
 *         Replica 2:
 *              Root = XYZ789
 *
 *    Iska matlab hai ki dono datasets mein kahin na kahin difference hai.
 *
 *         Root Hash Different
 *                |
 *                v
 *         Data Difference Exists
 *
 *    Ab hum poora dataset compare nahi karte.
 *
 *    Instead, tree ke lower levels compare karte hain.
 *
 *
 *         Root
 *        /    \
 *      H12    H34
 *
 *    Maan lo:
 *
 *       Replica 1:
 *          H12 = A
 *          H34 = B
 *
 *       Replica 2:
 *          H12 = A
 *          H34 = X
 *
 *    H12 same hai:
 *
 *         H12 == H12
 *
 *    Isliye left side ka data same hai.
 *
 *    H34 different hai:
 *
 *         H34 != H34
 *
 *    To hum sirf H34 ke children ko compare karenge.
 *
 *
 *         Root Different
 *              |
 *              v
 *           Compare
 *          /       \
 *       Same     Different
 *        |           |
 *        v           v
 *      Ignore    Go Deeper
 *                    |
 *                    v
 *              Find Different Bucket
 *
 *    Is process se hum quickly identify kar sakte hain ki
 *    kaunse data buckets different hain.
 *
 *
 *    Example:
 *
 *         Replica 1:
 *
 *              Root
 *             /    \
 *           H12    H34
 *          /  \    /  \
 *         H1  H2  H3  H4
 *
 *         Replica 2:
 *
 *              Root
 *             /    \
 *           H12    X34
 *          /  \    /  \
 *         H1  H2  X3  H4
 *
 *    Root different.
 *
 *    H12 same -> Left side same.
 *
 *    H34 different -> Check children.
 *
 *    H4 same -> Ignore.
 *
 *    H3 different -> B3 bucket has different data.
 *
 *    Ab synchronization mainly B3 ke data par ki ja sakti hai.
 *
 *
 *    Isliye humein millions of keys ko blindly compare nahi karna padta.
 *
 *
 *    Complete Anti-Entropy Flow:
 *
 *         Replica 1                  Replica 2
 *             |                          |
 *             v                          v
 *        Build/Compare              Build/Compare
 *        Merkle Tree                Merkle Tree
 *             |                          |
 *             v                          v
 *          Root H1                    Root H2
 *             |                          |
 *             +------------+-------------+
 *                          |
 *                     Compare Root
 *                          |
 *                    +-----+-----+
 *                    |           |
 *                  Same       Different
 *                    |           |
 *                    v           v
 *                 Nothing    Compare Child Hashes
 *                 to Sync          |
 *                                  v
 *                           Find Different Buckets
 *                                  |
 *                                  v
 *                           Synchronize Data
 *
 *
 *    Benefits:
 *
 *    -> Efficient Synchronization
 *
 *       Poore dataset ko compare karne ki zarurat nahi hoti.
 *       Sirf divergent buckets identify kiye jaate hain.
 *
 *    -> Lower Network Cost
 *
 *       Sirf differences ko synchronize karne se unnecessary
 *       data transfer reduce hota hai.
 *
 *    -> Scales to Large Systems
 *
 *       Large datasets aur millions of buckets ke saath bhi
 *       Merkle Trees efficiently differences identify karne
 *       mein help karte hain.
 *
 *    -> Eventual Consistency
 *
 *       Regular synchronization ke through divergent replicas
 *       eventually same state par converge kar sakti hain.
 *
 *
 *    Important Idea:
 *
 *    Synchronization cost ideally total dataset size ke proportional
 *    nahi honi chahiye.
 *
 *    Instead, cost primarily data differences par depend kare.
 *
 *         Small Difference
 *              |
 *              v
 *       Small Synchronization
 *
 *         Large Difference
 *              |
 *              v
 *       Larger Synchronization
 *
 *    Yaani synchronization cost ko differences ke size ke
 *    proportional rakhne ki koshish ki jaati hai, na ki
 *    poore dataset ke size ke proportional.
 *
 *
 *    Simple words mein:
 *
 *    Anti-Entropy Protocol replicas ke beech data ko synchronize
 *    karne ka mechanism hai.
 *
 *    Poora data compare karna expensive hota hai, isliye
 *    Merkle Tree ka use kiya ja sakta hai.
 *
 *    Merkle Tree mein:
 *
 *       -> Leaves = Data buckets ke hashes
 *       -> Internal Nodes = Child hashes ke hashes
 *       -> Root Hash = Entire dataset ka summary
 *
 *    Agar do replicas ka Root Hash same hai:
 *
 *         Root A == Root B
 *              |
 *              v
 *         Data is identical
 *
 *    Agar Root Hash different hai:
 *
 *         Root A != Root B
 *              |
 *              v
 *         Tree ko recursively compare karo
 *              |
 *              v
 *         Different buckets find karo
 *              |
 *              v
 *         Sirf divergent data synchronize karo
 *
 *
 *    Core Idea:
 *
 *         Replica A                    Replica B
 *             |                            |
 *             v                            v
 *        Merkle Tree                  Merkle Tree
 *             |                            |
 *             v                            v
 *          Root A                      Root B
 *             |                            |
 *             +------------+---------------+
 *                          |
 *                     Compare Hash
 *                          |
 *                    +-----+-----+
 *                    |           |
 *                  Same       Different
 *                    |           |
 *                    v           v
 *                 In Sync    Find Difference
 *                                  |
 *                                  v
 *                            Sync Buckets
 *                                  |
 *                                  v
 *                           Replicas Converge
 *
 *    Yaani Anti-Entropy + Merkle Trees ka main goal hai:
 *
 *    "Large distributed dataset ko efficiently compare karke
 *     sirf divergent data ko synchronize karna, taaki replicas
 *     eventually consistent state mein aa saken."
 */

In [ ]:
/**
 * Build Merkle Tree:
 *
 *    Ab hum step-by-step dekhenge ki Merkle Tree actually build
 *    kaise hota hai.
 *
 *    Maan lo hamare Key-Value Store mein key space:
 *
 *         1 se 12
 *
 *    Matlab possible keys hain:
 *
 *         1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12
 *
 *
 *    Step-1: Divide Key Space into Buckets
 *
 *    Sabse pehle complete key space ko chhote-chhote buckets
 *    mein divide karte hain.
 *
 *         Key Space: 1 - 12
 *
 *         +---------+---------+---------+---------+
 *         | Bucket 1| Bucket 2| Bucket 3| Bucket 4|
 *         +---------+---------+---------+---------+
 *         |  1 - 3  |  4 - 6  |  7 - 9  | 10 - 12|
 *         +---------+---------+---------+---------+
 *
 *    Yaani:
 *
 *         Bucket 1 -> Keys 1, 2, 3
 *         Bucket 2 -> Keys 4, 5, 6
 *         Bucket 3 -> Keys 7, 8, 9
 *         Bucket 4 -> Keys 10, 11, 12
 *
 *    Buckets ka purpose hai large dataset ko smaller sections
 *    mein divide karna, taaki baad mein hum identify kar saken
 *    ki difference exactly kis section mein hai.
 *
 *
 *    Step-2: Hash Each Key inside a Bucket
 *
 *    Ab har bucket ke andar ke individual keys ko ek
 *    uniform hashing method se hash karte hain.
 *
 *         Bucket 1:
 *
 *         Key 1 -> Hash(1) = H1
 *         Key 2 -> Hash(2) = H2
 *         Key 3 -> Hash(3) = H3
 *
 *         Bucket 2:
 *
 *         Key 4 -> Hash(4) = H4
 *         Key 5 -> Hash(5) = H5
 *         Key 6 -> Hash(6) = H6
 *
 *         ...and so on.
 *
 *    Example:
 *
 *         Bucket 1
 *            |
 *        +---+---+---+
 *        |   |   |   |
 *        v   v   v
 *       H1  H2  H3
 *
 *
 *    Step-3: Create One Hash Node per Bucket
 *
 *    Ab bucket ke andar ke saare key hashes ko combine karke
 *    ek single hash calculate karte hain.
 *
 *         Bucket 1:
 *
 *         H1
 *         H2
 *         H3
 *          \ | /
 *           \|/
 *        Hash(H1 + H2 + H3)
 *               |
 *               v
 *             B1
 *
 *    Yaani Bucket 1 ka ek single hash node banega:
 *
 *         B1 = Hash(H1 + H2 + H3)
 *
 *
 *    Similarly:
 *
 *         B2 = Hash(H4 + H5 + H6)
 *
 *         B3 = Hash(H7 + H8 + H9)
 *
 *         B4 = Hash(H10 + H11 + H12)
 *
 *
 *    Ab hamare paas har bucket ke liye ek hash node hai:
 *
 *         B1       B2       B3       B4
 *         |        |        |        |
 *         v        v        v        v
 *       Hash     Hash     Hash     Hash
 *
 *
 *    Step-4: Build the Tree Upwards
 *
 *    Ab bucket hashes ko use karke tree ko bottom-up
 *    build karte hain.
 *
 *         B1       B2       B3       B4
 *          \       /         \       /
 *           \     /           \     /
 *            B12                 B34
 *             \                   /
 *              \                 /
 *               +----- ROOT ----+
 *
 *    First:
 *
 *         B12 = Hash(B1 + B2)
 *
 *         B34 = Hash(B3 + B4)
 *
 *    Then:
 *
 *         ROOT = Hash(B12 + B34)
 *
 *
 *    Complete Merkle Tree:
 *
 *                     +-------------+
 *                     |     ROOT    |
 *                     +------+------+ 
 *                            |
 *                    +-------+-------+
 *                    |               |
 *                    v               v
 *                 +-----+         +-----+
 *                 | B12 |         | B34 |
 *                 +--+--+         +--+--+
 *                    |               |
 *                 +--+--+         +--+--+
 *                 |     |         |     |
 *                 v     v         v     v
 *                B1    B2        B3    B4
 *                |     |         |     |
 *              1-3   4-6       7-9   10-12
 *
 *
 *    Yaani tree bottom se top ki taraf build hota hai:
 *
 *         Keys
 *          |
 *          v
 *       Key Hashes
 *          |
 *          v
 *      Bucket Hashes
 *          |
 *          v
 *     Parent Hashes
 *          |
 *          v
 *        Root Hash
 *
 *
 *    Complete Example:
 *
 *         Key Space = 1 to 12
 *
 *         Bucket 1 -> 1, 2, 3
 *         Bucket 2 -> 4, 5, 6
 *         Bucket 3 -> 7, 8, 9
 *         Bucket 4 -> 10, 11, 12
 *
 *              |
 *              v
 *
 *         Hash each key
 *
 *         B1 = Hash(H1 + H2 + H3)
 *         B2 = Hash(H4 + H5 + H6)
 *         B3 = Hash(H7 + H8 + H9)
 *         B4 = Hash(H10 + H11 + H12)
 *
 *              |
 *              v
 *
 *         B12 = Hash(B1 + B2)
 *         B34 = Hash(B3 + B4)
 *
 *              |
 *              v
 *
 *         ROOT = Hash(B12 + B34)
 *
 *
 *    Ab poore dataset ko ek single Root Hash represent kar raha hai.
 *
 *    Agar kisi bucket ka data change hota hai:
 *
 *         Key Change
 *             |
 *             v
 *         Key Hash Change
 *             |
 *             v
 *         Bucket Hash Change
 *             |
 *             v
 *         Parent Hash Change
 *             |
 *             v
 *         Root Hash Change
 *
 *    Isi property ki wajah se do replicas ko efficiently compare
 *    kiya ja sakta hai.
 *
 *
 *    Simple words mein:
 *
 *    Merkle Tree build karne ke 4 main steps hain:
 *
 *       1. Key space ko buckets mein divide karo.
 *
 *       2. Har bucket ke individual keys ko hash karo.
 *
 *       3. Har bucket ke hashes ko combine karke ek bucket hash banao.
 *
 *       4. Bucket hashes ko recursively combine karke tree ko
 *          bottom-up root tak build karo.
 *
 *
 *    Core Idea:
 *
 *         Key Space: 1 - 12
 *                |
 *                v
 *          Divide into Buckets
 *                |
 *                v
 *       +------+------+------+------+
 *       | B1   | B2   | B3   | B4   |
 *       | 1-3  | 4-6  | 7-9  |10-12 |
 *       +------+------+------+------+
 *                |
 *                v
 *          Bucket Hashes
 *                |
 *                v
 *            B12    B34
 *               \  /
 *                \/
 *               ROOT
 *
 *    Yaani Merkle Tree ek hierarchical hash structure hai jo
 *    entire key space ko efficiently represent karta hai.
 */

In [ ]:
/**
 * Synchronization Process using Merkle Trees:
 *
 *    Ab humne Merkle Tree build karna samajh liya.
 *    Ab dekhte hain ki do replicas ke beech Merkle Tree ka use karke
 *    synchronization kaise hoti hai.
 *
 *    Maan lo same data ki do replicas hain:
 *
 *         Replica 1                  Replica 2
 *             |                          |
 *             v                          v
 *        Merkle Tree                Merkle Tree
 *
 *    Dono replicas same key partitioning aur same hashing scheme
 *    use karte hain.
 *
 *
 *    Step-1: Each Replica Builds its Own Merkle Tree
 *
 *    Replica 1 apne data ka Merkle Tree banata hai.
 *
 *         Replica 1
 *              |
 *              v
 *         +---------+
 *         | Merkle  |
 *         |  Tree   |
 *         +---------+
 *              |
 *              v
 *           Root A
 *
 *    Replica 2 bhi apne data ka Merkle Tree banata hai.
 *
 *         Replica 2
 *              |
 *              v
 *         +---------+
 *         | Merkle  |
 *         |  Tree   |
 *         +---------+
 *              |
 *              v
 *           Root B
 *
 *    Important:
 *
 *       -> Same key partitioning honi chahiye.
 *       -> Same hashing scheme honi chahiye.
 *
 *    Tabhi dono Merkle Trees ko correctly compare kiya ja sakta hai.
 *
 *
 *    Step-2: Compare Root Hashes
 *
 *    Sabse pehle dono replicas ke Root Hash compare karte hain.
 *
 *         Replica 1              Replica 2
 *             |                      |
 *             v                      v
 *          Root A                 Root B
 *             |                      |
 *             +----------+-----------+
 *                        |
 *                  Compare Hashes
 *                        |
 *                   +----+----+
 *                   |         |
 *                 Match    Mismatch
 *
 *
 *    Case 1: Root Hash Match
 *
 *         Root A == Root B
 *
 *    Iska matlab hai dono replicas ka corresponding data identical hai.
 *
 *         Replica 1 -> Root A
 *                       =
 *         Replica 2 -> Root A
 *
 *              |
 *              v
 *       No Synchronization
 *          Required
 *
 *
 *    Case 2: Root Hash Mismatch
 *
 *         Root A != Root B
 *
 *    Iska matlab hai replicas ke data mein kahin na kahin difference hai.
 *
 *         Replica 1 -> Root A
 *                       !=
 *         Replica 2 -> Root B
 *
 *              |
 *              v
 *       Data Difference Exists
 *
 *    Lekin abhi humein nahi pata ki difference exactly kahan hai.
 *
 *
 *    Step-3: Recursive Comparison
 *
 *    Root mismatch hone ke baad poora dataset compare nahi karte.
 *
 *    Instead, root ke child hashes compare karte hain.
 *
 *
 *              Root
 *             /    \
 *           H12    H34
 *
 *    Replica 1:
 *
 *           H12 = A
 *           H34 = B
 *
 *    Replica 2:
 *
 *           H12 = A
 *           H34 = X
 *
 *
 *    Compare:
 *
 *         H12:
 *
 *            A == A
 *             |
 *             v
 *          No Difference
 *
 *         H34:
 *
 *            B != X
 *             |
 *             v
 *          Difference
 *
 *    Isliye hum sirf H34 wale subtree mein drill down karenge.
 *
 *
 *    Step-4: Drill Down into Different Subtrees
 *
 *    Maan lo H34 ke children hain:
 *
 *              H34
 *             /   \
 *           H3     H4
 *
 *    Ab H3 aur H4 compare karenge.
 *
 *         H3:
 *            Same
 *
 *         H4:
 *            Different
 *
 *    To ab sirf H4 ke subtree mein further drill down karenge.
 *
 *
 *         Root
 *          |
 *       +--+--+
 *       |     |
 *      H12   H34
 *      |      |
 *    Same   Different
 *             |
 *             v
 *           H3  H4
 *           |    |
 *         Same Different
 *                 |
 *                 v
 *          Inconsistent Bucket
 *
 *
 *    Is process ko recursively repeat kiya jaata hai jab tak
 *    exact inconsistent bucket identify na ho jaaye.
 *
 *
 *    Step-5: Identify Inconsistent Buckets
 *
 *    Finally humein pata chal jaata hai ki kaunse buckets ke
 *    hashes different hain.
 *
 *         Replica 1:
 *
 *            Bucket 1 -> Same
 *            Bucket 2 -> Same
 *            Bucket 3 -> DIFFERENT
 *            Bucket 4 -> Same
 *
 *         Replica 2:
 *
 *            Bucket 1 -> Same
 *            Bucket 2 -> Same
 *            Bucket 3 -> DIFFERENT
 *            Bucket 4 -> Same
 *
 *
 *    Ab poora dataset exchange karne ki zarurat nahi hai.
 *
 *    Sirf inconsistent bucket ko exchange aur repair karna hai.
 *
 *         Bucket 3
 *            |
 *            v
 *       Compare Data
 *            |
 *            v
 *       Find Missing /
 *       Outdated Data
 *            |
 *            v
 *       Synchronize
 *            |
 *            v
 *       Repair Replica
 *
 *
 *    Complete Synchronization Flow:
 *
 *         Replica 1                  Replica 2
 *             |                          |
 *             v                          v
 *        Build Tree                 Build Tree
 *             |                          |
 *             v                          v
 *          Root A                     Root B
 *             |                          |
 *             +------------+-------------+
 *                          |
 *                    Compare Roots
 *                          |
 *                    +-----+-----+
 *                    |           |
 *                  Match      Mismatch
 *                    |           |
 *                    v           v
 *                  Done     Compare Children
 *                                  |
 *                                  v
 *                         Find Different Subtree
 *                                  |
 *                                  v
 *                         Drill Down Recursively
 *                                  |
 *                                  v
 *                       Identify Bad Buckets
 *                                  |
 *                                  v
 *                         Exchange Data
 *                                  |
 *                                  v
 *                         Repair Replicas
 *
 *
 *    Example:
 *
 *         Total Dataset:
 *
 *         +------+------+------+------+
 *         | B1   | B2   | B3   | B4   |
 *         +------+------+------+------+
 *           Same   Same  Diff   Same
 *
 *    Agar sirf B3 different hai:
 *
 *         Total Dataset = 4 Buckets
 *
 *         Synchronize:
 *
 *             B3 only
 *
 *         NOT:
 *
 *             B1 + B2 + B3 + B4
 *
 *
 *    Key Insight:
 *
 *    Merkle Tree synchronization ka main benefit ye hai ki
 *    sirf different data transfer karna padta hai,
 *    poora dataset nahi.
 *
 *
 *         Without Merkle Tree:
 *
 *         Replica A
 *             |
 *             v
 *       Compare Entire Data
 *             |
 *             v
 *       Transfer Large Dataset
 *
 *
 *         With Merkle Tree:
 *
 *         Replica A              Replica B
 *             |                      |
 *             v                      v
 *        Merkle Tree            Merkle Tree
 *             |                      |
 *             +----------+-----------+
 *                        |
 *                   Compare Root
 *                        |
 *                        v
 *                 Drill Down Only
 *                 Where Hash Differs
 *                        |
 *                        v
 *                Find Different Bucket
 *                        |
 *                        v
 *                  Transfer Only
 *                  Different Data
 *
 *
 *    Simple words mein:
 *
 *    Har replica apne data ka Merkle Tree banata hai.
 *
 *    Pehle Root Hash compare hota hai:
 *
 *       Root same:
 *          -> Data same
 *          -> Synchronization ki zarurat nahi.
 *
 *       Root different:
 *          -> Data kahin different hai.
 *          -> Child hashes compare karo.
 *          -> Sirf different subtree mein drill down karo.
 *          -> Finally inconsistent bucket identify karo.
 *          -> Sirf us bucket ka data exchange aur repair karo.
 *
 *
 *    Core Idea:
 *
 *         Build Merkle Trees
 *                |
 *                v
 *         Compare Root Hash
 *                |
 *          +-----+-----+
 *          |           |
 *        Match      Mismatch
 *          |           |
 *          v           v
 *        Done      Compare Children
 *                      |
 *                      v
 *                Drill Down
 *                      |
 *                      v
 *             Find Different Bucket
 *                      |
 *                      v
 *               Synchronize Data
 *
 *    Yaani Merkle Trees ka main purpose hai:
 *
 *    "Efficiently identify karna ki replicas ke data mein
 *     difference exactly kahan hai, taaki sirf wahi data
 *     synchronize kiya jaaye."
 */

In [ ]:
/**
 * System Architecture:
 *
 *    Ab hum Distributed Key-Value Store ka complete system architecture
 *    samajhte hain.
 *
 *    Hamara system client ko do simple APIs provide karega:
 *
 *       -> get(key)
 *       -> put(key, value)
 *
 *    Client ko internally kitne servers hain, data kahan stored hai,
 *    ya replication kaise ho rahi hai — ye sab details pata hone ki
 *    zarurat nahi hai.
 *
 *
 *    1. Simple Client APIs:
 *
 *       Client
 *          |
 *          +----> put(key, value)
 *          |
 *          +----> get(key)
 *
 *       Example:
 *
 *          put("user:123", "Mohammad")
 *
 *          get("user:123")
 *
 *    Client simply key ke saath operation request karta hai.
 *
 *
 *    2. Coordinator Node:
 *
 *    Client request cluster ke kisi node par jaati hai.
 *    Jo node request receive karta hai, wo us request ke liye
 *    Coordinator ka role perform kar sakta hai.
 *
 *
 *         Client
 *            |
 *            | get(key) / put(key, value)
 *            v
 *       +-------------+
 *       | Coordinator |
 *       +------+------+
 *              |
 *              v
 *        Find responsible
 *        nodes for key
 *
 *    Coordinator:
 *
 *       -> Key ko hash karta hai.
 *       -> Consistent Hash Ring par key ki position find karta hai.
 *       -> Responsible node(s) identify karta hai.
 *       -> Replicas ke saath request coordinate karta hai.
 *       -> Quorum ke according responses/ACKs collect karta hai.
 *       -> Final response client ko return karta hai.
 *
 *
 *    3. Consistent Hashing Ring:
 *
 *    Cluster ke nodes ek Consistent Hashing Ring par organized hote hain.
 *
 *
 *                    +-------- S1 --------+
 *                  /                       \
 *                S4                         S2
 *                 |                         |
 *                  \                       /
 *                    +-------- S3 --------+
 *
 *    Key ko hash karke ring par ek position milti hai.
 *    Us position ke basis par responsible node identify hota hai.
 *
 *
 *    Example:
 *
 *         Key
 *          |
 *          v
 *      Hash Function
 *          |
 *          v
 *      Hash Position
 *          |
 *          v
 *      Consistent Hash Ring
 *          |
 *          v
 *      Responsible Node
 *
 *
 *    4. Data Replication:
 *
 *    High availability ke liye data ki multiple replicas
 *    different nodes par maintain ki jaati hain.
 *
 *         Key
 *          |
 *          v
 *       Hash Ring
 *          |
 *          +----> S1 -> Replica 1
 *          |
 *          +----> S2 -> Replica 2
 *          |
 *          +----> S3 -> Replica 3
 *
 *    Agar S1 fail ho jaaye, to S2/S3 ke paas data ki copies
 *    available reh sakti hain.
 *
 *
 *    5. Decentralized Design:
 *
 *    System mein koi single master/server nahi hota jo
 *    entire cluster ko control kare.
 *
 *    Multiple nodes independently requests handle kar sakte hain.
 *
 *
 *         +------+     +------+     +------+
 *         |  S1  |<--->|  S2  |<--->|  S3  |
 *         +------+     +------+     +------+
 *            ^            ^            ^
 *            |            |            |
 *            +------------+------------+
 *
 *    Har node:
 *
 *       -> Requests handle kar sakta hai.
 *       -> Coordinator ka role perform kar sakta hai.
 *       -> Data store kar sakta hai.
 *       -> Replication mein participate karta hai.
 *       -> Failure detection mein participate kar sakta hai.
 *
 *
 *    6. Automatic Scaling:
 *
 *    Consistent Hashing ki wajah se cluster mein nodes
 *    relatively easily add/remove kiye ja sakte hain.
 *
 *
 *         Before:
 *
 *          S1 ---- S2 ---- S3
 *
 *         Add S4:
 *
 *          S1 ---- S2 ---- S3 ---- S4
 *
 *    New node add hone par affected partitions/keys ko
 *    new node par move kiya ja sakta hai.
 *
 *    Poore dataset ko redistribute karne ki zarurat nahi hoti.
 *
 *
 *         More Traffic
 *              |
 *              v
 *         Add More Nodes
 *              |
 *              v
 *       Data Repartitioning
 *              |
 *              v
 *         More Capacity
 *
 *
 *    7. No Single Point of Failure:
 *
 *    Architecture ka important goal hai ki koi single server
 *    entire system ko bring down na kar sake.
 *
 *
 *         Client
 *            |
 *       +----+----+----+
 *       |    |    |    |
 *       v    v    v    v
 *      S1   S2   S3   S4
 *
 *    Agar S2 fail ho jaata hai:
 *
 *      S1 -> Healthy
 *      S2 -> FAILED
 *      S3 -> Healthy
 *      S4 -> Healthy
 *
 *    Baaki nodes system ko continue kar sakte hain.
 *
 *
 *    8. Equal Responsibilities:
 *
 *    Kisi ek master node par saari responsibility nahi hoti.
 *
 *    Nodes distributed way mein:
 *
 *       -> Requests handle karte hain.
 *       -> Data store karte hain.
 *       -> Replicas maintain karte hain.
 *       -> Failure detection mein participate karte hain.
 *       -> Coordination perform kar sakte hain.
 *
 *
 *    Complete System Architecture:
 *
 *
 *                         +----------------+
 *                         |     Client     |
 *                         +-------+--------+
 *                                 |
 *                         get(key) / put()
 *                                 |
 *                                 v
 *                         +----------------+
 *                         |   Coordinator  |
 *                         |    Node       |
 *                         +-------+--------+
 *                                 |
 *                                 v
 *                     +-----------------------+
 *                     | Consistent Hash Ring  |
 *                     +-----------------------+
 *                       /        |        \
 *                      v         v         v
 *                  +------+  +------+  +------+
 *                  |  S1  |  |  S2  |  |  S3  |
 *                  +--+---+  +--+---+  +--+---+
 *                     |         |         |
 *                     +---------+---------+
 *                               |
 *                         Replication
 *                               |
 *                         Multiple Copies
 *
 *
 *    Request Flow:
 *
 *       1. Client sends:
 *
 *             put(key, value)
 *
 *       2. Request kisi node par arrive hoti hai.
 *
 *       3. Receiving node Coordinator ban jaata hai.
 *
 *       4. Coordinator key ko hash karta hai.
 *
 *       5. Consistent Hash Ring se responsible replicas
 *          identify karta hai.
 *
 *       6. Request required replicas ko forward karta hai.
 *
 *       7. Quorum ke according acknowledgements/responses
 *          collect karta hai.
 *
 *       8. Coordinator final response client ko return karta hai.
 *
 *
 *    High-Level Flow:
 *
 *         Client
 *           |
 *           v
 *       Coordinator
 *           |
 *           v
 *       Hash Key
 *           |
 *           v
 *     Consistent Hash Ring
 *           |
 *       +---+---+---+
 *       |   |   |   |
 *       v   v   v   v
 *      S1  S2  S3  S4
 *       |   |   |
 *       +---+---+
 *           |
 *           v
 *       Replication
 *           |
 *           v
 *       Quorum
 *           |
 *           v
 *       Response
 *           |
 *           v
 *        Client
 *
 *
 *    Simple words mein:
 *
 *    Distributed Key-Value Store ka architecture ek cluster of
 *    equal nodes par based hai.
 *
 *    Client sirf simple get(key) aur put(key, value) APIs use karta hai.
 *
 *    Request receive karne wala node Coordinator ka role perform
 *    karke key ko Consistent Hashing Ring par locate karta hai.
 *
 *    Data multiple nodes par replicate hota hai taaki system
 *    highly available aur fault tolerant rahe.
 *
 *    Decentralized architecture ki wajah se nodes add/remove
 *    karke system ko automatically scale kiya ja sakta hai.
 *
 *    Aur kyunki kisi ek master node par complete dependency nahi hai,
 *    system mein Single Point of Failure avoid kiya ja sakta hai.
 *
 *
 *    Core Idea:
 *
 *         Simple APIs
 *              |
 *              v
 *         Coordinator
 *              |
 *              v
 *      Consistent Hash Ring
 *              |
 *        +-----+-----+-----+
 *        |     |     |     |
 *        v     v     v     v
 *       S1    S2    S3    S4
 *        |     |     |
 *        +-----+-----+
 *              |
 *              v
 *        Data Replication
 *              |
 *              v
 *          High Availability
 *
 *    Yaani complete system ka main idea hai:
 *
 *    "Data ko multiple nodes par intelligently partition aur
 *     replicate karo, requests ko distributed way mein coordinate karo,
 *     aur kisi single node par entire system ko dependent mat rakho."
 */

In [ ]:
/**
 * Write Path:
 *
 *    Ab hum Distributed Key-Value Store ka Write Path samajhte hain.
 *
 *    Write Path ka matlab hai:
 *
 *    "Jab client put(key, value) request bhejta hai,
 *     to data system ke andar kaise process aur store hota hai?"
 *
 *
 *    Example:
 *
 *         Client
 *            |
 *            | put(key, value)
 *            v
 *       Coordinator
 *            |
 *            v
 *       Write Path
 *
 *
 *    Step-1: Client sends Write Request
 *
 *    Client ek write request bhejta hai:
 *
 *         put("user:123", "Mohammad")
 *
 *    Request cluster ke kisi node par arrive hoti hai.
 *
 *         +--------+
 *         | Client |
 *         +---+----+
 *             |
 *             | put(key, value)
 *             v
 *        +-----------+
 *        |Coordinator|
 *        +-----+-----+
 *              |
 *              v
 *         Write Path
 *
 *    Coordinator responsible replicas identify karta hai aur
 *    write ko process karne ke liye aage bhejta hai.
 *
 *
 *    Step-2: Commit Log
 *
 *    Write ko directly sirf memory mein store karna risky hai.
 *
 *    Maan lo:
 *
 *         Client -> Write
 *                    |
 *                    v
 *                Memory
 *                    |
 *                    X
 *                 Server Crash
 *
 *    Agar data sirf memory mein tha, to crash ke baad data lose ho sakta hai.
 *
 *    Isliye write ko pehle disk par Commit Log mein append kiya jaata hai.
 *
 *
 *         Write Request
 *              |
 *              v
 *         +------------+
 *         | Commit Log |
 *         |   (Disk)   |
 *         +------------+
 *
 *    Commit Log mein write sequentially append hota hai.
 *
 *         Write 1 -> Write 2 -> Write 3 -> Write 4 -> ...
 *
 *    Agar server crash ho jaaye, to system commit log ko replay
 *    karke lost in-memory state recover kar sakta hai.
 *
 *    Iska main purpose hai:
 *
 *         Durability
 *
 *    Yaani successfully recorded write crash ke baad bhi recover
 *    kiya ja sakta hai.
 *
 *
 *    Step-3: Memory Cache / Memtable
 *
 *    Commit Log ke baad write ko ek in-memory data structure
 *    mein bhi store kiya jaata hai, jise commonly Memtable kaha jaata hai.
 *
 *         Write
 *           |
 *           +---------> Commit Log
 *           |               |
 *           |               v
 *           |            Disk
 *           |
 *           +---------> Memtable
 *                           |
 *                           v
 *                         Memory
 *
 *    Memtable ka benefit hai fast access because data RAM mein hota hai.
 *
 *         Memtable:
 *
 *         +----------------------+
 *         | Key       | Value    |
 *         +----------------------+
 *         | user:101  | Alice    |
 *         | user:102  | Bob      |
 *         | user:103  | Charlie  |
 *         +----------------------+
 *
 *    Isse recently written data ko quickly process/read kiya ja sakta hai.
 *
 *
 *    Step-4: Flush to Disk
 *
 *    Memtable memory mein limited size tak data store karta hai.
 *
 *    Jab Memtable ek predefined threshold reach karta hai:
 *
 *         Memtable
 *             |
 *             | Size Threshold Reached
 *             v
 *         Flush to Disk
 *
 *    Memory mein data ko disk par write kiya jaata hai.
 *
 *
 *    Step-5: SSTable
 *
 *    Memtable ka data disk par ek SSTable ke form mein store hota hai.
 *
 *    SSTable ka full form hai:
 *
 *         Sorted String Table
 *
 *    SSTable ek immutable, sorted file hoti hai jisme
 *    key-value pairs stored hote hain.
 *
 *
 *         Memtable
 *             |
 *             | Flush
 *             v
 *         +----------------------+
 *         |       SSTable        |
 *         |----------------------|
 *         | key1 -> value1       |
 *         | key2 -> value2       |
 *         | key3 -> value3       |
 *         | key4 -> value4       |
 *         +----------------------+
 *                  Disk
 *
 *    Immutable ka matlab hai:
 *
 *    Ek baar SSTable disk par create hone ke baad us file ko
 *    directly modify nahi kiya jaata.
 *
 *    New updates ke liye new SSTables create ho sakti hain.
 *
 *
 *    Complete Write Flow:
 *
 *         +--------+
 *         | Client |
 *         +---+----+
 *             |
 *             | put(key, value)
 *             v
 *         +-----------+
 *         |Coordinator|
 *         +-----+-----+
 *               |
 *               v
 *         +------------+
 *         | Commit Log |
 *         |    Disk    |
 *         +-----+------+
 *               |
 *               v
 *         +------------+
 *         |  Memtable  |
 *         |   Memory   |
 *         +-----+------+
 *               |
 *          Threshold?
 *               |
 *               v
 *         +------------+
 *         |  SSTable   |
 *         |    Disk    |
 *         +------------+
 *
 *
 *    Example:
 *
 *         Client:
 *
 *         put(user:123, "Mohammad")
 *
 *                |
 *                v
 *
 *         Commit Log:
 *
 *         user:123 -> Mohammad
 *
 *                |
 *                v
 *
 *         Memtable:
 *
 *         user:123 -> Mohammad
 *
 *                |
 *                | Memtable Full
 *                v
 *
 *         SSTable:
 *
 *         user:123 -> Mohammad
 *
 *
 *    Ab data ke paas do important protections hain:
 *
 *         Commit Log
 *             |
 *             v
 *         Durability
 *
 *         Memtable
 *             |
 *             v
 *         Fast Access
 *
 *    Aur jab Memtable flush hota hai:
 *
 *         Memtable
 *             |
 *             v
 *         SSTable
 *             |
 *             v
 *         Persistent Disk Storage
 *
 *
 *    Key Insight:
 *
 *    Write Path mein writes ko sequential way mein append karna
 *    generally fast hota hai.
 *
 *         Write
 *           |
 *           v
 *      Sequential Append
 *           |
 *           v
 *       Commit Log
 *
 *    Isliye system ko har update ke liye random disk locations
 *    par directly modify karne ki zarurat nahi hoti.
 *
 *
 *    Commit Log + Memtable ka combination:
 *
 *         Commit Log
 *              |
 *              +--> Durability
 *
 *         Memtable
 *              |
 *              +--> Fast Memory Access
 *
 *         SSTable
 *              |
 *              +--> Persistent Sorted Storage
 *
 *
 *    Yaani:
 *
 *         Commit Log + Memtable + SSTable
 *                    |
 *                    v
 *          Durability + Performance
 *
 *
 *    Simple words mein:
 *
 *    Client jab put(key, value) request bhejta hai,
 *    to write pehle Commit Log mein append hoti hai taaki
 *    crash ke case mein data recover kiya ja sake.
 *
 *    Uske baad data Memtable mein store hota hai taaki
 *    fast memory access possible ho.
 *
 *    Jab Memtable ek certain size threshold reach karta hai,
 *    uska data disk par SSTable ke form mein flush ho jaata hai.
 *
 *    SSTable immutable aur sorted key-value file hoti hai.
 *
 *
 *    Core Idea:
 *
 *         Client
 *           |
 *           v
 *       Coordinator
 *           |
 *           v
 *       Commit Log
 *           |
 *           | Durability
 *           v
 *        Memtable
 *           |
 *           | Fast Access
 *           v
 *       Size Threshold
 *           |
 *           v
 *        SSTable
 *           |
 *           v
 *          Disk
 *
 *    Is design ka main goal hai:
 *
 *    "Writes ko fast rakhna aur saath hi crashes ke against
 *     data ko durable banana."
 */

In [ ]:
/**
 * Read Path:
 *
 *    Ab hum Distributed Key-Value Store ka Read Path samajhte hain.
 *
 *    Read Path ka matlab hai:
 *
 *    "Jab client get(key) request bhejta hai, to system data ko
 *     efficiently kaise find karke client ko return karta hai?"
 *
 *
 *    Example:
 *
 *         Client
 *            |
 *            | get(key)
 *            v
 *       Coordinator
 *            |
 *            v
 *         Read Path
 *
 *
 *    Step-1: Client sends Read Request
 *
 *    Client ek read request bhejta hai:
 *
 *         get("user:123")
 *
 *    Request cluster ke kisi node par arrive hoti hai.
 *
 *         +--------+
 *         | Client |
 *         +---+----+
 *             |
 *             | get(key)
 *             v
 *        +-----------+
 *        |Coordinator|
 *        +-----+-----+
 *              |
 *              v
 *          Read Path
 *
 *    Coordinator key ke responsible replica nodes identify karke
 *    data retrieve karne ka process start karta hai.
 *
 *
 *    Step-2: Memory Cache / Memtable Check
 *
 *    Sabse pehle system memory mein check karta hai ki requested
 *    key available hai ya nahi.
 *
 *         get(user:123)
 *               |
 *               v
 *          Memory Check
 *               |
 *          +----+----+
 *          |         |
 *        Found    Not Found
 *          |         |
 *          v         v
 *       Return    Disk Lookup
 *
 *    Agar data Memtable/memory mein mil jaata hai:
 *
 *         Memory
 *            |
 *            v
 *       user:123 -> Mohammad
 *            |
 *            v
 *          Client
 *
 *    Data immediately return kiya ja sakta hai.
 *
 *    Isse disk I/O avoid hota hai aur read latency low rehti hai.
 *
 *
 *    Example:
 *
 *         get(user:123)
 *              |
 *              v
 *          Memtable
 *              |
 *              v
 *       user:123 found
 *              |
 *              v
 *          Return Value
 *
 *
 *    Step-3: Cache Miss / Disk Lookup
 *
 *    Agar requested key memory mein nahi milti:
 *
 *         get(user:123)
 *              |
 *              v
 *          Memtable
 *              |
 *              X
 *          Cache Miss
 *              |
 *              v
 *         Disk Lookup
 *
 *    Ab system ko SSTables mein key search karni padegi.
 *
 *    Lekin problem ye hai ki disk par multiple SSTables ho sakti hain.
 *
 *         Disk
 *          |
 *      +---+---+---+---+
 *      |   |   |   |   |
 *      v   v   v   v   v
 *     S1  S2  S3  S4  S5
 *
 *    Agar hum har SSTable ko blindly search karein:
 *
 *         Search S1
 *         Search S2
 *         Search S3
 *         Search S4
 *         Search S5
 *
 *    To unnecessary disk I/O increase ho sakta hai.
 *
 *    Is problem ko reduce karne ke liye Bloom Filter use kiya
 *    ja sakta hai.
 *
 *
 *    Step-4: Bloom Filter
 *
 *    Bloom Filter ek probabilistic data structure hai jo quickly
 *    check karne mein help karta hai ki key kisi SSTable mein
 *    "possibly present" hai ya "definitely not present".
 *
 *
 *         Key
 *          |
 *          v
 *      Bloom Filter
 *          |
 *       +--+--+
 *       |     |
 *    Definitely  Possibly
 *      Not       Present
 *       |           |
 *       v           v
 *     Skip       Check SSTable
 *
 *
 *    Important:
 *
 *    Bloom Filter agar bolta hai:
 *
 *         "Key definitely nahi hai"
 *
 *    To hum us SSTable ko safely skip kar sakte hain.
 *
 *    Agar Bloom Filter bolta hai:
 *
 *         "Key possibly hai"
 *
 *    To actual SSTable ko check karna padega.
 *
 *    Bloom Filter false positive de sakta hai,
 *    lekin false negative nahi dena chahiye.
 *
 *
 *    Example:
 *
 *         get(user:123)
 *              |
 *              v
 *         Bloom Filters
 *              |
 *       +------+------+------+------+
 *       |      |      |      |      |
 *       v      v      v      v      v
 *      S1     S2     S3     S4     S5
 *       |      |      |      |      |
 *      No     Yes     No     Yes     No
 *       |      |      |      |      |
 *       X      v      X      v      X
 *              |             |
 *              v             v
 *          Check S2       Check S4
 *
 *    Yaani sirf relevant SSTables ko access karna padega.
 *
 *
 *    Step-5: Retrieve Data from SSTable
 *
 *    Bloom Filter ne potentially relevant SSTables identify kar di.
 *
 *         Bloom Filter
 *              |
 *              +----> SSTable 2
 *              |
 *              +----> SSTable 4
 *
 *    Ab system in SSTables ko actually search karta hai.
 *
 *         SSTable 2
 *              |
 *              v
 *       user:123 -> Mohammad
 *
 *    Data milne ke baad result client ko return kiya jaata hai.
 *
 *
 *    Step-6: Return Result
 *
 *         SSTable
 *            |
 *            v
 *       Retrieved Value
 *            |
 *            v
 *       Coordinator
 *            |
 *            v
 *          Client
 *
 *    Example:
 *
 *         get(user:123)
 *              |
 *              v
 *          "Mohammad"
 *
 *
 *    Complete Read Flow:
 *
 *         +--------+
 *         | Client |
 *         +---+----+
 *             |
 *             | get(key)
 *             v
 *        +-----------+
 *        |Coordinator|
 *        +-----+-----+
 *              |
 *              v
 *         Memory Check
 *              |
 *         +----+----+
 *         |         |
 *       Found    Not Found
 *         |         |
 *         v         v
 *       Return   Bloom Filter
 *                   |
 *                   v
 *              Relevant SSTables
 *                   |
 *                   v
 *              Disk Lookup
 *                   |
 *                   v
 *              Retrieve Value
 *                   |
 *                   v
 *                Client
 *
 *
 *    Example:
 *
 *         Client:
 *
 *         get(user:123)
 *
 *                |
 *                v
 *
 *         +---------------+
 *         |   Memtable    |
 *         +-------+-------+
 *                 |
 *              Not Found
 *                 |
 *                 v
 *
 *         +---------------+
 *         | Bloom Filters |
 *         +-------+-------+
 *                 |
 *          +------+------+
 *          |             |
 *          v             v
 *       SSTable 2     SSTable 4
 *          |             |
 *          +------+------+
 *                 |
 *                 v
 *           Find Key
 *                 |
 *                 v
 *            Return Value
 *
 *
 *    Key Insight:
 *
 *    Read Path ko fast banane ke liye do important techniques
 *    use ki ja sakti hain:
 *
 *       1. In-Memory Cache / Memtable
 *          -> Frequently/recently available data ko fast memory
 *             se return kiya ja sakta hai.
 *
 *       2. Bloom Filter
 *          -> Unnecessary SSTables ko skip karke disk I/O reduce
 *             kiya ja sakta hai.
 *
 *
 *    Simple words mein:
 *
 *    Client jab get(key) request bhejta hai, request Coordinator
 *    ke paas jaati hai.
 *
 *    Sabse pehle memory/Memtable mein key search hoti hai.
 *
 *    Agar key mil gayi:
 *
 *         -> Immediately value return.
 *
 *    Agar key nahi mili:
 *
 *         -> Bloom Filters check kiye jaate hain.
 *         -> Jo SSTables key ko contain nahi kar sakti,
 *            unhe skip kar diya jaata hai.
 *         -> Sirf relevant SSTables ko disk par search kiya jaata hai.
 *         -> Value milne ke baad client ko return ki jaati hai.
 *
 *
 *    Core Idea:
 *
 *         Client
 *           |
 *           v
 *       Coordinator
 *           |
 *           v
 *       Memory Check
 *           |
 *       +---+---+
 *       |       |
 *     Found   Miss
 *       |       |
 *       v       v
 *     Return  Bloom Filter
 *               |
 *               v
 *        Relevant SSTables
 *               |
 *               v
 *          Disk Lookup
 *               |
 *               v
 *          Return Value
 *
 *    Yaani:
 *
 *       In-Memory Cache -> Fast Reads
 *       Bloom Filter    -> Less Unnecessary Disk I/O
 *
 *    Dono techniques milkar large-scale Key-Value Store ke
 *    read performance ko improve karne mein help karti hain.
 */

In [ ]:
/**
 * Summary:
 *
 *    Ab humne Distributed Key-Value Store ke major problems aur
 *    unke solutions ko ek jagah summarize karte hain.
 *
 *    Goal ye tha ki system:
 *
 *       -> Bahut large data store kar sake.
 *       -> Highly available ho.
 *       -> Easily scale ho sake.
 *       -> Different capacity wale servers support kar sake.
 *       -> Failures ko handle kar sake.
 *       -> Consistency ko requirement ke according tune kar sake.
 *
 *
 *    +-------------------------------------------------------------------+
 *    | Problem                         | Solution                         |
 *    +-------------------------------------------------------------------+
 *    | 1. Ability to store big data   | Consistent Hashing               |
 *    |                                | -> Data ko multiple servers      |
 *    |                                |    par distribute karta hai.     |
 *    +--------------------------------+----------------------------------+
 *    | 2. High availability of reads | Data Replication                 |
 *    |                                | -> Multiple copies available.   |
 *    |                                |                                  |
 *    |                                | Multi-DC Setup                  |
 *    |                                | -> Different data centers mein  |
 *    |                                |    replicas maintain karna.      |
 *    +--------------------------------+----------------------------------+
 *    | 3. High availability writes  | Versioning + Conflict            |
 *    |                                | Resolution using Vector Clocks |
 *    |                                | -> Concurrent updates ko        |
 *    |                                |    detect/resolve karna.         |
 *    +--------------------------------+----------------------------------+
 *    | 4. Dataset Partitioning       | Consistent Hashing               |
 *    |                                | -> Large dataset ko partitions  |
 *    |                                |    mein divide karta hai.       |
 *    +--------------------------------+----------------------------------+
 *    | 5. Incremental Scalability    | Consistent Hashing               |
 *    |                                | -> Servers add/remove karne par |
 *    |                                |    minimum data movement.       |
 *    +--------------------------------+----------------------------------+
 *    | 6. Heterogeneity              | Consistent Hashing +            |
 *    |                                | Virtual Nodes                   |
 *    |                                | -> Different capacity wale      |
 *    |                                |    servers support hote hain.   |
 *    +--------------------------------+----------------------------------+
 *    | 7. Tunable Consistency        | Quorum Consensus                |
 *    |                                | -> R, W aur N ke through        |
 *    |                                |    consistency/latency tune.    |
 *    +--------------------------------+----------------------------------+
 *    | 8. Temporary Failure          | Sloppy Quorum + Hinted         |
 *    |                                | Handoff                         |
 *    |                                | -> Healthy nodes temporarily   |
 *    |                                |    requests handle karte hain.  |
 *    +--------------------------------+----------------------------------+
 *    | 9. Permanent Failure          | Merkle Tree + Anti-Entropy      |
 *    |                                | -> Divergent replicas ko        |
 *    |                                |    efficiently synchronize.     |
 *    +--------------------------------+----------------------------------+
 *    | 10. Data Center Outage        | Cross-Data-Center Replication   |
 *    |                                | -> Different DCs mein replicas  |
 *    |                                |    maintain karna.              |
 *    +--------------------------------+----------------------------------+
 *
 *
 *    Ab in concepts ko ek complete picture mein dekho:
 *
 *
 *                         +----------------+
 *                         |     Client     |
 *                         +-------+--------+
 *                                 |
 *                          get() / put()
 *                                 |
 *                                 v
 *                         +----------------+
 *                         |  Coordinator  |
 *                         +-------+--------+
 *                                 |
 *                                 v
 *                      +-----------------------+
 *                      | Consistent Hash Ring  |
 *                      +-----------------------+
 *                        /        |        \
 *                       v         v         v
 *                     S1         S2         S3
 *                      |          |          |
 *                      +----------+----------+
 *                                 |
 *                          Data Replication
 *                                 |
 *                    +------------+------------+
 *                    |                         |
 *                    v                         v
 *                Data Center 1            Data Center 2
 *                    |                         |
 *                 Replicas                  Replicas
 *
 *
 *    Failures ke case mein:
 *
 *         Temporary Failure
 *                |
 *                v
 *         Sloppy Quorum
 *                |
 *                v
 *         Hinted Handoff
 *                |
 *                v
 *         Node Recovers
 *
 *
 *         Permanent Failure / Divergence
 *                |
 *                v
 *         Merkle Tree
 *                |
 *                v
 *         Anti-Entropy
 *                |
 *                v
 *         Replica Synchronization
 *
 *
 *    Consistency ke case mein:
 *
 *         Multiple Replicas
 *                |
 *                v
 *          Quorum Consensus
 *                |
 *          +-----+-----+
 *          |           |
 *        Read        Write
 *          |           |
 *          v           v
 *          R           W
 *                |
 *                v
 *             N Replicas
 *
 *
 *    Conflicting Versions ke case mein:
 *
 *         Concurrent Writes
 *                |
 *                v
 *          Vector Clocks
 *                |
 *                v
 *        Conflict Detection
 *                |
 *                v
 *        Conflict Resolution
 *
 *
 *    Simple words mein:
 *
 *    Distributed Key-Value Store mein ek single technique
 *    saari problems solve nahi karti.
 *
 *    Har problem ke liye ek specific mechanism use hota hai:
 *
 *       -> Consistent Hashing
 *          Data Partitioning + Scaling + Heterogeneity
 *
 *       -> Data Replication
 *          High Availability
 *
 *       -> Multi-Data-Center Replication
 *          Data Center Failure Protection
 *
 *       -> Quorum Consensus
 *          Tunable Consistency
 *
 *       -> Vector Clocks
 *          Versioning + Conflict Detection
 *
 *       -> Sloppy Quorum + Hinted Handoff
 *          Temporary Failure Handling
 *
 *       -> Merkle Trees + Anti-Entropy
 *          Replica Synchronization after Divergence
 *
 *
 *    Final Mental Model:
 *
 *         Large Data
 *            |
 *            v
 *      Consistent Hashing
 *            |
 *            v
 *       Data Partitioning
 *            |
 *            v
 *       Data Replication
 *            |
 *            v
 *      High Availability
 *            |
 *       +----+----+
 *       |         |
 *       v         v
 *     Quorum   Vector Clocks
 *       |         |
 *       v         v
 *   Consistency  Conflict
 *       |       Resolution
 *       |
 *       +-------------------+
 *                           |
 *                           v
 *                   Failure Handling
 *                    /            \
 *                   v              v
 *           Temporary Failure   Permanent Failure
 *                   |              |
 *                   v              v
 *            Sloppy Quorum    Merkle Tree
 *                   |              |
 *                   v              v
 *          Hinted Handoff     Anti-Entropy
 *                   |              |
 *                   +------+-------+
 *                          |
 *                          v
 *                  Replica Convergence
 *
 *
 *    Overall idea:
 *
 *    "Data ko intelligently partition aur replicate karo,
 *     consistency ko quorum/versioning ke through manage karo,
 *     aur failures ke baad replicas ko synchronization mechanisms
 *     ke through eventually consistent state mein lao."
 */